# `LinearMHDDriftkineticCC`: a hands-on tutorial

**Energetic ions (5D drift-kinetic PIC) coupled to linear ideal MHD (FEEC) through the current-coupling scheme**

You already know how to run `LinearMHD`, for example the ITPA TAE case in
`examples/LinearMHD/itpa_tae_benchmark/params_TAE_benchmark.py`. This notebook builds on that. Everything you
know about `domain`, `equil`, `grid`, `DerhamOptions`, `EnvironmentOptions`, `Time`, and the FEEC
perturbations carries over unchanged. The focus here is what is new:

1. **What changes** between `LinearMHD` and `LinearMHDDriftkineticCC` (species, propagators, scalars, constructor).
2. **Just enough physics** to know what each term and propagator does, and what the parameter $\varepsilon$ means.
3. **The kinetic species**:
   * the `GyroMaxwellian2D` background: every argument, profiles, and the `B0` trap
   * `set_markers(...)`: loading, importance sampling, control variate (δf), boundaries, sorting, and saving
4. **The seven propagators and their `Options`**: what each solves, which knobs matter, and recommended settings.
5. **Live experiments you run in a cell**: sampling quality, δf weights, a pusher-accuracy test, passing and trapped orbits in the ITPA torus, a small coupled "sandbox" run, and its post-processing.
6. **A production template** for TAE + energetic ions (written to disk and ready for `srun`), a smoke test, and an analysis toolkit (energy exchange, growth rates) applied to your existing `LinearMHD` baseline run.
7. **A pitfalls checklist and cheat sheet.**

> **Runtime.** Everything runs on one core. Approximate cost of the heavier cells on an idle node: orbit run ≈ 2 min,
> pusher test ≈ 10 s, equilibrium test ≈ 1 min, sandbox run ≈ 1 min, TAE smoke test ≈ 3 min (several times longer on a busy login node;
> set `RUN_SMOKE_TEST = False` to skip it). All output goes to `./kinetic_tutorial_runs/`.
> Run the cells top to bottom, because later sections reuse helpers from earlier ones.

> The original 8-cell notebook was saved as `Kinetic_Particles_original_backup.ipynb`.

In [ ]:
import os, time, logging, importlib.util
import numpy as np
import matplotlib.pyplot as plt
import h5py

from struphy import set_logging_level
set_logging_level(logging.WARNING)          # INFO prints every option object; WARNING keeps the notebook readable

from struphy import (
    BaseUnits, DerhamOptions, EnvironmentOptions, Simulation, Time,
    domains, equils, grids, perturbations, maxwellians,
    # --- new for kinetic species ---
    LoadingParameters, WeightsParameters, BoundaryParameters,
    SortingParameters, SavingParameters, BinningPlot,
)
from struphy.models import LinearMHD, LinearMHDDriftkineticCC

# parameter objects used inside propagator Options (not exported at the top level of struphy)
from struphy.linear_algebra.solver import SolverParameters, DiscreteGradientSolverParameters
from struphy.pic.accumulation.filter import FilterParameters
from struphy.ode.utils import ButcherTableau

RUNS = os.path.abspath("kinetic_tutorial_runs")
os.makedirs(RUNS, exist_ok=True)

# plotting: fixed colour order, thin lines, recessive grid
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25, "lines.linewidth": 1.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=C),
})
print("output folder:", RUNS)

---
## 1  From `LinearMHD` to `LinearMHDDriftkineticCC`: what changes

| | `LinearMHD` (you know this) | `LinearMHDDriftkineticCC` (new) |
|---|---|---|
| **Constructor** | `LinearMHD(base_units)` | `LinearMHDDriftkineticCC(base_units, mhd_mass_number=1.0, hot_charge_number=1, hot_mass_number=1.0, hot_epsilon=None, turn_off=(None,))` |
| **FEEC species** | `em_fields.b_field` (Hdiv), `mhd.density/pressure` (L2), `mhd.velocity` (Hdiv) | **identical** |
| **Kinetic species** | – | `energetic_ions.var`, a `PICVariable` in space **`Particles5D`** with coordinates $(\eta_1,\eta_2,\eta_3,v_\parallel,\mu)$ |
| **Propagators** | `shear_alf`, `mag_sonic` | `push_bxe`, `push_parallel`, `shearalfen_cc5d`, `magnetosonic`, `cc5d_density`, `cc5d_gradb`, `cc5d_curlb` |
| **Scalars** | `en_U, en_p, en_B, en_tot, …` | `en_U, en_p, en_B, `**`en_fv, en_fB`**`, en_tot, `**`n_lost_particles`** |
| **Extra set-up** | – | `model.energetic_ions.set_markers(...)` (marker numerics) and `model.energetic_ions.var.add_background(GyroMaxwellian2D(...))` (mandatory), plus optionally `add_initial_condition(...)` |
| **Time step** | limited by the MHD solvers | also by the particle pushers and by the cost of the (dt-dependent) Schur solve |

Key idea: **the MHD part keeps the same variables and equations**, with an added energetic-particle (EP) current
in the momentum equation. You set up the MHD fields exactly as before, including your TAE `TorusModesSin/Cos`
perturbation of `mhd.velocity`. The new work is describing the EP distribution and choosing numerics for its markers.

The next cell prints the two models side by side, straight from the objects.

In [ ]:
def describe(model):
    print(f"=== {type(model).__name__}")
    for sname, sp in model.species.items():
        print(f"  species {sname:15s}", {k: v.space for k, v in sp.variables.items()})
    print("  propagators (in splitting order):")
    for name, prop in vars(model.propagators).items():
        vars_ = {k.lstrip('_'): getattr(v, 'space', None) for k, v in vars(prop.variables).items()}
        print(f"    {name:16s} {type(prop).__name__:32s} updates {vars_}")
    print("  scalars:", list(model.scalars.dct))
    print()

describe(LinearMHD())
describe(LinearMHDDriftkineticCC())

**Reading the output:**

* The order of `vars(model.propagators)` **is the Lie–Trotter splitting order** used in each time step
  (`Time(split_algo="LieTrotter")`). `"Strang"` is also available and symmetrizes the sequence.
* `shearalfen_cc5d` replaces `shear_alf`. It is the same shear-Alfvén block, plus the EP magnetization term.
  `magnetosonic` is the same propagator as `mag_sonic`, just under another name.
* The three `cc5d_*` propagators update `u` (and some also update the markers). They contain the EP → MHD feedback.
* `push_bxe` and `push_parallel` only move markers (the EP guiding centers).

---
## 2  Just enough physics

### 2.1  The drift-kinetic phase space
A marker is a **guiding center**, not a particle. The fast gyration has been averaged out, which leaves five coordinates:

$$
(\boldsymbol\eta, v_\parallel, \mu),\qquad v_\parallel=\mathbf v\cdot\mathbf b_0,\qquad \mu=\frac{v_\perp^2}{2|\mathbf B_0|}\ \ (\text{normalized}).
$$

The magnetic moment $\mu$ is an **exact invariant** of the model. No propagator ever changes column 4 of the marker array.
The distribution $f_h(\boldsymbol\eta,v_\parallel,\mu,t)$ obeys

$$
\partial_t f_h+\underbrace{\frac{v_\parallel\mathbf B^*-\mathbf b_0\times\mathbf E^*}{B^*_\parallel}}_{\dot{\mathbf X}}\cdot\nabla f_h
+\underbrace{\frac{1}{\varepsilon}\frac{\mathbf B^*\cdot\mathbf E^*}{B^*_\parallel}}_{\dot v_\parallel}\partial_{v_\parallel}f_h=0,
\qquad
\mathbf B^*=\mathbf B+\varepsilon v_\parallel\nabla\times\mathbf b_0,\quad
\mathbf E^*=-\tilde{\mathbf U}\times\mathbf B-\varepsilon\mu\nabla|\mathbf B| .
$$

The terms in $\dot{\mathbf X}$ are, from left to right: **parallel streaming** ($v_\parallel\mathbf B/B^*_\parallel$),
the **curvature drift** ($\varepsilon v_\parallel^2\nabla\times\mathbf b_0$), the **∇B drift** ($\varepsilon\mu\,\mathbf b_0\times\nabla|B|$), and the **E×B drift** in the MHD wave field ($\tilde{\mathbf U}_\perp$).
The $\dot v_\parallel$ equation contains the **mirror force** $-\mu\,\mathbf b\cdot\nabla|B|$, which creates trapped (banana) orbits.

### 2.2  Current coupling: how EPs act back on the MHD fluid
The MHD momentum equation gains an EP current term:

$$
\rho_0\partial_t\tilde{\mathbf U}=\dots\;+\;\frac{A_h}{A_b}\Big[\tfrac1\varepsilon n_{gc}\tilde{\mathbf U}-\tfrac1\varepsilon\mathbf J_{gc}-\nabla\times\mathbf M_{gc}\Big]\times\mathbf B .
$$

Here $\mathbf J_{gc}=\int f_h\,\dot{\mathbf X}\,B^*_\parallel\,dv_\parallel d\mu$ is the guiding-center current, and
$\mathbf M_{gc}=-\int f_h B^*_\parallel\mu\mathbf b_0$ is the magnetization. All three moments are **accumulated from the markers onto the FEEC grid** at every step.
Energy moves between the fluid and the EPs through these terms, and the model conserves the total `en_tot` exactly.

### 2.3  Who does what: propagator ↔ physics map

| propagator | physics it advances | updates |
|---|---|---|
| `push_bxe` (`PushGuidingCenterBxEstar`) | $\dot{\mathbf X}=\mathbf E^*\times\mathbf b_0/B^*_\parallel$ with $\mathbf E^*=-\varepsilon\mu\nabla|B|$: **∇B drift** | markers $\boldsymbol\eta$ |
| `push_parallel` (`PushGuidingCenterParallel`) | $\dot{\mathbf X}=v_\parallel\mathbf B^*/B^*_\parallel$, $\dot v_\parallel\propto\mathbf B^*\cdot\mathbf E^*$: **streaming + curvature drift + mirror force** | markers $\boldsymbol\eta, v_\parallel$ |
| `cc5d_gradb` (`CurrentCoupling5DGradB`) | EP **∇B-drift current** pushes $\mathbf U$; markers are E×B-advected by $\tilde{\mathbf U}\times\mathbf B$ | $\mathbf u$, markers $\boldsymbol\eta$ |
| `cc5d_curlb` (`CurrentCoupling5DCurlb`) | EP **curvature current** $\propto v_\parallel^2\nabla\times\mathbf b_0$ pushes $\mathbf U$; markers are accelerated in $v_\parallel$ | $\mathbf u$, markers $v_\parallel$ |
| `cc5d_density` (`CurrentCoupling5DDensity`) | the $\frac1\varepsilon n_{gc}(1-B_\parallel/B^*_\parallel)\tilde{\mathbf U}\times\mathbf B$ term (EP "Lorentz" density term) | $\mathbf u$ |
| `shearalfen_cc5d` (`ShearAlfvenCurrentCoupling5D`) | shear-Alfvén wave $(\mathbf u,\mathbf b)$ **plus the EP magnetization** $\nabla\times\mathbf M_{gc}$ | $\mathbf u,\mathbf b$ |
| `magnetosonic` (`Magnetosonic`) | compressional part, the same as in `LinearMHD` | $n,\mathbf u,p$ |

### 2.4  What drives a TAE unstable (and what you need in the set-up)
* **Resonance.** EPs exchange energy with a TAE at frequency $\omega$ when their transit frequency matches it, which roughly means $v_\parallel\approx v_A$ and the sideband $v_\parallel\approx v_A/3$. The EP thermal speed must therefore be comparable to $v_A$.
* **Drive.** The source of free energy is the **radial gradient** of the EP density (the $\omega_*$ drive), located where the mode sits. A flat EP profile gives no drive.
* **Damping.** Landau damping $\propto\partial f/\partial E$ competes with the drive. For a Maxwellian it always damps.
* **Consequence for a homogeneous slab.** There is no curvature, no ∇B, and no gradient, so the EPs cannot exchange energy with the wave. That is why the slab sandbox in Section 7 is for learning the mechanics, and the torus is where the physics happens.

### 2.5  Units and the parameter $\varepsilon$
The velocity scale of this model is the **bulk Alfvén speed** $\hat v=\hat v_A=\hat B/\sqrt{\mu_0 A_b m_p\hat n}$, so EP velocities are measured in units of $\hat v_A$.
The new parameter is

$$
\varepsilon=\frac{1}{\hat\Omega_{c,h}\,\hat t}=\frac{A_h m_p}{Z_h e\,\hat B}\,\frac{\hat v_A}{\hat x}
=\frac{A_h}{Z_h}\frac{m_p}{e\,\hat x\sqrt{\mu_0 A_b m_p \hat n}} .
$$

$\hat B$ cancels out, so $\varepsilon$ is the **ion inertial length divided by $\hat x$** (times $A_h/Z_h$ and a mass-ratio factor).
It sets the size of the drifts, the orbit widths ($\rho_h\sim\varepsilon v_\perp/B$), and the strength of the coupling ($1/\varepsilon$ terms). It is computed automatically from `BaseUnits` and the charge and mass numbers.
You can override it with `hot_epsilon=...`, for example to scan orbit width at fixed everything else. The next cell checks the scalings numerically.

In [ ]:
def eps_of(bu=BaseUnits(), **kw):
    m = LinearMHDDriftkineticCC(base_units=bu, **kw)
    return m.energetic_ions.equation_params.epsilon, m.units.v

rows = [
    ("default BaseUnits()",               BaseUnits(),               {}),
    ("B = 3 T",                           BaseUnits(B=3.0),          {}),
    ("n = 0.2 (2e19 m^-3)",               BaseUnits(n=0.2),          {}),
    ("x = 2 m",                           BaseUnits(x=2.0),          {}),
    ("hot deuterons (A_h=2)",             BaseUnits(),               dict(hot_mass_number=2.0)),
    ("hot alphas (A_h=4, Z_h=2)",         BaseUnits(),               dict(hot_mass_number=4.0, hot_charge_number=2)),
    ("override hot_epsilon=0.05",         BaseUnits(),               dict(hot_epsilon=0.05)),
]
print(f"{'case':32s} {'epsilon':>10s} {'v_hat = v_A [m/s]':>20s}")
for name, bu, kw in rows:
    e, v = eps_of(bu, **kw)
    print(f"{name:32s} {e:10.4f} {v:20.3e}")

Note that **B does not change ε** but does change $\hat v$, and that ε ∝ $\hat n^{-1/2}$, ∝ $1/\hat x$, and ∝ $A_h/Z_h$.

> **Watch out:** in your TAE set-up `BaseUnits()` means $\hat B=1\,$T, while the equilibrium has $B_0=3$. The *local*
> Alfvén speed is then $v_A(r)=|B_0(r)|/\sqrt{n_0(r)}$ in code units (with $\hat n=10^{20}\,\mathrm{m}^{-3}$ and the bulk mass $A_b$), which is about 3, not 1. The resonance condition $v_\parallel\approx v_A$ must be evaluated with this local value (Section 3.4).

Converting physical EP parameters into code units:

In [ ]:
E_CHARGE, M_PROTON = 1.602176634e-19, 1.67262192369e-27

def ep_code_params(model, T_keV):
    '''Thermal speed sqrt(T/m) of the energetic ions in units of v_hat (what GyroMaxwellian2D wants),
    plus epsilon and the orbit-width estimate rho_h = eps * vth / B.'''
    A_h = model.energetic_ions.mass_number
    vth_phys = np.sqrt(T_keV * 1e3 * E_CHARGE / (A_h * M_PROTON))
    vth = vth_phys / model.units.v
    eps = model.energetic_ions.equation_params.epsilon
    return dict(vth_phys=vth_phys, vth=vth, eps=eps)

for T, A in [(100, 1.0), (400, 1.0), (400, 2.0)]:
    m = LinearMHDDriftkineticCC(hot_mass_number=A)
    p = ep_code_params(m, T)
    print(f"T={T:4d} keV, A_h={A}:  vth={p['vth_phys']:.2e} m/s -> vth_code={p['vth']:.3f},  eps={p['eps']:.4f},"
          f"  rho_h(B=3) ~ {p['eps']*p['vth']/3:.4f} (in units of x_hat)")

---
## 3  The kinetic background: `GyroMaxwellian2D`

```python
maxwellians.GyroMaxwellian2D(
    n        = (1.0, None),   # density         (background, perturbation)
    u_para   = (0.0, None),   # parallel shift  -> beams
    u_perp   = (0.0, None),   # must stay 0     (mu >= 0 has no shift)
    vth_para = (1.0, None),   # parallel thermal speed  sqrt(T_par/m)  in units of v_A
    vth_perp = (1.0, None),   # perpendicular thermal speed sqrt(T_perp/m)
    volume_form = True,       # keep True (markers work with volume forms)
    B0 = 2.0,                 # !!! |B| used to convert mu -> v_perp.  Default 2.0 is almost never what you want
    uniform_on_disc = False,  # multiply n by 2*eta1 (uniform density on a disc-like logical domain)
)
```

Its analytic form is

$$
f=n(\boldsymbol\eta)\;\frac{1}{\sqrt{2\pi}\,v_{th\parallel}}e^{-\frac{(v_\parallel-u_\parallel)^2}{2v_{th\parallel}^2}}\;
\frac{1}{v_{th\perp}^2}e^{-\mu B_0/v_{th\perp}^2}\;\times\;\underbrace{B_0}_{\text{volume form}} .
$$

Every moment is a **tuple `(background, perturbation)`**:
* `background`: a float, or a **callable of η** (a spatial profile).
* `perturbation`: `None` or any `perturbations.*` object (evaluated in **logical** coordinates η). It is only used in the initial condition (see 3.5).

### 3.1  Look at it in $(v_\parallel,\mu)$

In [ ]:
def eval_on_vmu(f, v, mu, eta=(0.5, 0.5, 0.5)):
    '''Evaluate a 5D background on a (v_par, mu) grid at one point in space (flat evaluation).'''
    V, MU = np.meshgrid(v, mu, indexing="ij")
    e = [np.full(V.size, x) for x in eta]
    return f(*e, V.ravel(), MU.ravel()).reshape(V.shape)

v  = np.linspace(-4, 4, 201)
mu = np.linspace(0, 2, 151)

f_iso  = maxwellians.GyroMaxwellian2D(n=(1.0, None), vth_para=(1.0, None), vth_perp=(1.0, None), B0=3.0)
f_beam = maxwellians.GyroMaxwellian2D(n=(1.0, None), u_para=(1.5, None), vth_para=(0.4, None), vth_perp=(0.6, None), B0=3.0)

fig, axs = plt.subplots(1, 2, figsize=(11, 3.8), layout="constrained")
for ax, f, title in zip(axs, [f_iso, f_beam], ["isotropic: vth_para = vth_perp = 1", "beam: u_para=1.5, vth_para=0.4, vth_perp=0.6"]):
    F = eval_on_vmu(f, v, mu)
    cs = ax.contourf(v, mu, F.T, levels=20, cmap="Blues")
    ax.set(xlabel=r"$v_\parallel$ [$v_A$]", ylabel=r"$\mu$", title=title)
    fig.colorbar(cs, ax=ax)
plt.show()

# moment accessors (evaluated on eta arrays)
e = np.full(3, 0.5)
print("n   =", f_beam.n(e, e, e))
print("u   =", f_beam.u(e, e, e), "  (u_para, u_perp)")
print("vth =", f_beam.vth(e, e, e), "  (vth_para, vth_perp)")

Left plot: 
Along $v_\parallel$: a Gaussian centred at 0. Particles are equally likely to travel either way along the field, with a width of about ±1 $v_A$.  Along μ: the distribution is largest at μ = 0 and decays exponentially. It falls by a factor e at $\mu = v_{th\perp}^2/B_0 = 1/3$, which is why almost nothing appears above μ ≈ 1. Dome-shaped contours: each contour is a curve where $\tfrac12 v_\parallel^2 + \mu B_0$ is constant. That quantity is the particle energy $E = \tfrac12 v_\parallel^2 + \mu|B|$. So in the isotropic case, the contour lines are lines of constant energy

Right Plot:
almost every particle travels in the +B direction at about 1.5 $v_A$. This is a beam, e.g. from neutral-beam injection. Squashed in μ: the e-folding value is $0.6^2/3 = 0.12$, so particles have little perpendicular energy. The beam is mostly parallel.
Peak ≈ 8.4, much higher than the left panel: the total particle number is still n = 1, but it is packed into a much smaller patch of velocity space, so the peak has to be higher.

### 3.2  The `B0` trap (important)
$\mu$ is not a velocity. The conversion to $v_\perp$ uses the **local** field: $v_\perp^2 = 2\mu|B(\boldsymbol\eta)|$.
The Maxwellian uses the `B0` you pass, so in terms of $v_\perp$ it reads

$$
f\propto \exp\!\Big(-\frac{v_\perp^2}{2v_{th\perp}^2}\cdot\frac{B_0^{\rm param}}{|B(\boldsymbol\eta)|}\Big)
\;\Rightarrow\; v_{th\perp}^{\rm eff}=v_{th\perp}\sqrt{|B(\boldsymbol\eta)|/B_0^{\rm param}} .
$$

With the **default `B0=2.0`** and your equilibrium $|B|\approx3$, the perpendicular temperature is silently 50 % too high.
In a torus $|B|$ varies from about 2.7 to 3.3, so for a truly isotropic Maxwellian everywhere, pass the equilibrium's field
strength as a callable: **`B0=equil.absB0`**. For a slab (`HomogenSlab(B0z=1)`), use `B0=1.0`.

In [ ]:
vperp = np.linspace(0, 5, 300)
B_local = 3.0
fig, ax = plt.subplots(figsize=(6.5, 3.6), layout="constrained")
for B0p, lab in [(2.0, "B0=2.0 (default)"), (3.0, "B0=3.0 (= local |B|)")]:
    vth_eff = 1.0 * np.sqrt(B_local / B0p)
    ax.plot(vperp, vperp / vth_eff**2 * np.exp(-vperp**2 / (2 * vth_eff**2)), label=f"{lab}: vth_perp,eff={vth_eff:.2f}")
ax.set(xlabel=r"$v_\perp$ [$v_A$]", ylabel=r"$f(v_\perp)$ (2D-normalized)", title="Same vth_perp=1, different B0 parameter (local |B|=3)")
ax.legend(); plt.show()

- If B0 equals the real |B| where the particle sits, you get exactly the temperature you asked for.
- If B0 is wrong, the translation between μ and v⊥ is wrong, and the particles actually get a different perpendicular temperature. Struphy gives no warning.

An analogy: you set a price in euros, but the shop converts it with the wrong exchange rate. You pay a different amount from what you intended, and nothing tells you.

Reading the plot

Both curves use the same input, vth_perp = 1. The real field is |B| = 3.
Blue (B0 = 2, the default): the effective thermal speed is sqrt(3/2) = 1.22. The curve shifts right and spreads out, meaning the perpendicular temperature is 1.22² ≈ 1.5 times too high.

### 3.3  Sums of Maxwellians: core plus beam, or two populations
Backgrounds can be **added with `+`**. The result is a `SumKineticBackground`, and it is accepted wherever a single one is.
Your old template's `maxwellian_1 + maxwellian_2` did this.

In [ ]:
core = maxwellians.GyroMaxwellian2D(n=(1.0, None), vth_para=(1.0, None), vth_perp=(1.0, None), B0=3.0)
beam = maxwellians.GyroMaxwellian2D(n=(0.2, None), u_para=(2.5, None), vth_para=(0.3, None), vth_perp=(0.3, None), B0=3.0)
f_sum = core + beam
print(type(f_sum).__name__)

# marginal in v_par: integrate over mu numerically
F = eval_on_vmu(f_sum, v, mu)
fig, ax = plt.subplots(figsize=(6.5, 3.4), layout="constrained")
ax.plot(v, np.trapezoid(F, mu, axis=1), label="core + beam")
ax.plot(v, np.trapezoid(eval_on_vmu(core, v, mu), mu, axis=1), "--", label="core only")
ax.set(xlabel=r"$v_\parallel$", ylabel=r"$\int f\,d\mu$", title="Bump-on-tail in $v_\\parallel$")
ax.legend(); plt.show()

The shape is called bump-on-tail, and it is the textbook example of a distribution that can drive a wave unstable. The idea is Landau resonance:

- A wave moving at some phase speed mostly exchanges energy with particles moving at about the same speed. It's like surfers catching a wave.
- Particles slightly slower than the wave get pushed forward: they take energy from the wave.
- Particles slightly faster than the wave get held back: they give energy to the wave.
- The net result depends on which group is larger, and that is set by the slope of f at the wave speed:
  - Slope negative (fewer fast particles, like the dashed Maxwellian everywhere at v_par > 0): the wave loses energy. This is Landau damping.
  - Slope positive (the rising side of the bump, v_par ≈ 1.8 to 2.5): more faster particles than slower ones, so the wave gains energy. This is instability.
  
If the beam injection is set at the wavespeed then we can purposely drive the instability!

### 3.4  Spatial profiles and the ITPA torus
A moment can be a callable of the logical coordinates. **Struphy calls it in two different ways**:
* with **one** `(N, 3)` array of marker positions (flat evaluation, used when weights are computed), and
* with **three** arrays `(eta1, eta2, eta3)` from a meshgrid (used by plots and projections).

The helper below makes any `f(eta1)` safe for both calls. For `HollowTorus`, $r = a_1 + (a_2-a_1)\eta_1$.

The EP density is given **in units of $\hat n$**, the same units as the bulk `equil.n0`. The ratio
$n_{EP}/n_{bulk}$ is therefore what you set. The profile below has its steepest gradient at $r_0=0.5$, where the TAE sits, which is the kind of profile the ITPA benchmark uses.

In [ ]:
def eta1_profile(f_of_eta1):
    '''Wrap f(eta1) so it works with both of Struphy's calling conventions.'''
    def fun(*etas):
        e1 = etas[0][:, 0] if len(etas) == 1 else etas[0]
        return f_of_eta1(e1)
    return fun

# --- the ITPA-like torus (same as your LinearMHD TAE run) ---
a1 = 0.1
tae_domain = domains.HollowTorus(a1=a1, a2=1.0, R0=10.0, sfl=False, pol_period=1, tor_period=6)
tae_equil = equils.AdhocTorus(a=1.0, R0=10.0, B0=3.0, q_kind=0, p_kind=1, q0=1.71, q1=1.87,
                              p1=0.95, p2=0.05, beta=0.0018)
tae_equil.domain = tae_domain            # Simulation(...) does this for you; needed here for stand-alone evaluation

def r_of_eta1(e1):  return a1 + (1.0 - a1) * e1

def n_ep_of_r(r, n0=0.0072, r0=0.5, delta=0.2, L=0.3):
    '''EP density with maximum gradient at r0 (tanh step on a log scale).'''
    return n0 * np.exp(-delta / L * np.tanh((r - r0) / delta))

n_ep = eta1_profile(lambda e1: n_ep_of_r(r_of_eta1(e1)))

# evaluate equilibrium on the outboard midplane (theta=0) with flat (N,3) evaluation
e1 = np.linspace(0, 1, 200)
pts = np.stack([e1, 0 * e1, 0 * e1], axis=1)
r = r_of_eta1(e1)
n_bulk = tae_equil.n0(pts)
absB = tae_equil.absB0(pts)
absB_in = tae_equil.absB0(np.stack([e1, 0 * e1 + 0.5, 0 * e1], axis=1))
vA_loc = absB / np.sqrt(n_bulk)          # local Alfven speed in code units (bulk mass A_b=1)

fig, axs = plt.subplots(1, 3, figsize=(14, 3.5), layout="constrained")
axs[0].plot(r, n_ep(pts), label="n_EP (callable)")
axs[0].set(xlabel="r", ylabel=r"$n_{EP}$ [$\hat n$]", title="EP density profile")
ax2 = axs[0].twinx(); ax2.plot(r, n_ep(pts) / n_bulk, color=C[1], ls="--"); ax2.set_ylabel(r"$n_{EP}/n_{bulk}$", color=C[1]); ax2.grid(False)
axs[1].plot(r, absB, label=r"outboard ($\theta=0$)"); axs[1].plot(r, absB_in, label=r"inboard ($\theta=\pi$)")
axs[1].set(xlabel="r", ylabel="|B0|", title="|B0| varies: use B0=equil.absB0"); axs[1].legend()
axs[2].plot(r, vA_loc, label=r"$v_A(r)=|B_0|/\sqrt{n_0}$"); axs[2].plot(r, vA_loc / 3, label=r"$v_A/3$ sideband")
axs[2].set(xlabel="r", ylabel="code units", title="Resonant parallel velocities"); axs[2].legend()
plt.show()
print(f"at r=0.5:  n_bulk={np.interp(0.5, r, n_bulk):.3f},  |B|={np.interp(0.5, r, absB):.3f},  v_A={np.interp(0.5, r, vA_loc):.3f}")

plot 1:
Why the steepest drop is at r = 0.5: this is the most important physics in the set-up. A Maxwellian EP population damps waves in velocity space (the slope argument from 3.3), so the TAE drive has to come from the radial gradient of the EPs. Intuitively:

- There are more EPs inside than outside, so they carry free energy, like water held behind a dam.
- The TAE perturbs the EP orbits and lets some move outward. Moving down the gradient releases energy, and the wave picks it up.
- This only works where the wave actually is. Near r = 0.5 is where it sits: with q(r) = 1.71 + 0.16 r², q = 1.75 at r = 0.5, and 1.75 is the q value where the TAE gap forms for your n = 6 mode. So the steepest gradient is placed exactly on the mode.

A flat EP profile would give no drive. A gradient far frg either, because the wave isn't there.

plot 2:
B falls as 1/R

plot 3:

What a "resonance" means here: an EP exchanges energy efficiently with the TAE only if it keeps pace with the wave, i.e. it meets the wave in the same phase orbit after orbit

- v_par ≈ v_A: the main resonance.
- v_par ≈ v_A/3: the sideband. It comes from the toroidal geometry, because the wave's structure mixes two neighbouring poloidal harmonics.

at r = 0.5:  n_bulk = 0.80,  |B| = 2.86,  v_A = 3.20   →   sideband ≈ 1.07
This plot tells you how fast your EPs need to be.

**Choosing `vth`.** Compare the EP $v_\parallel$ distribution with the resonances. A Maxwellian with
$v_{th}\approx1$ has almost no particles at $v_\parallel\approx v_A\approx3$, so it would barely interact
with the main resonance. To study the drive you need $v_{th}\gtrsim v_A/2$. For the ITPA-like values
(400 keV, from `ep_code_params` above) you get $v_{th}\approx 2$–$3$. The next cell shows the fraction of EPs near each resonance for a few choices.

In [ ]:
vA0 = float(np.interp(0.5, r, vA_loc))
vv = np.linspace(-12, 12, 2001)
fig, ax = plt.subplots(figsize=(7.5, 3.6), layout="constrained")
for i, vth in enumerate([1.0, 2.0, 3.0]):
    g = np.exp(-vv**2 / (2 * vth**2)) / np.sqrt(2 * np.pi) / vth
    ax.plot(vv, g, color=C[i], label=f"vth={vth}")
    # integrate the +c and -c windows separately (one trapezoid over both would also integrate the gap between them)
    near = lambda c: sum(np.trapezoid(g[m], vv[m]) for m in (np.abs(vv - c) < 0.1 * c, np.abs(vv + c) < 0.1 * c))
    print(f"vth={vth}:  fraction within ±10% of |v_par|=v_A: {near(vA0):.3f},   of v_A/3: {near(vA0/3):.3f}")
for c, ls in [(vA0, "-"), (vA0 / 3, ":")]:
    for s in (-1, 1):
        ax.axvline(s * c, color="0.4", ls=ls, lw=1)
ax.set(xlabel=r"$v_\parallel$", ylabel="pdf", title=r"EP $v_\parallel$ distribution vs resonances (solid $v_A$, dotted $v_A/3$)")
ax.legend(); plt.show()

I understand vth like temperature, for a MB distribution it is basically the variance.

Reading each curve

- vth = 1 (blue): the curve is almost zero at ±3.2. Only 0.4% of EPs are near the main resonance, so they are nearly invisible to it. The sideband is well populated (9%), but it is the weaker interaction. This population is too cold to drive the TAE properly.
- vth = 2 (orange): about 7% at each resonance.
- vth = 3 (green): 9.4% near v_A, which is the most of the three, and fewer near the sideband.

There is a simple rule behind this. For a Gaussian, the share of particles at a given speed c is largest when vth ≈ c. If vth is too small, nobody reaches c. If vth is too large, the particles spread so thin that few are at c. The main resonance is c ≈ 3.2, so vth ≈ 3 is close to the best case.

### 3.5  Background, initial condition, perturbations: the rules

```python
model.energetic_ions.var.add_background(f0)            # MANDATORY. Used for the control variate (δf) and for diagnostics
model.energetic_ions.var.add_initial_condition(f_init) # OPTIONAL. If omitted, f_init = f0
```

* Kinetic perturbations are **not** added with `add_perturbation(...)`, which is how FEEC variables do it. Instead, they are put **inside a moment tuple** of the initial condition:
  `GyroMaxwellian2D(n=(n_ep, perturbations.TorusModesCos(...)), ...)`.
* For a TAE study, you usually perturb the **MHD velocity**, as in your `LinearMHD` run, and leave `f_init = f0`.
  The EP response then develops self-consistently.
* The background must be a function that is (approximately) **an equilibrium of the unperturbed guiding-center motion**.
  A $n(r)$ Maxwellian with $B_0=|B(\boldsymbol\eta)|$ is not an exact equilibrium: $n(r)$ is not a function of the constants of motion, so markers drift slowly and create a small δf even without a wave. This is acceptable for linear studies with δf. For exactness, `maxwellians.CanonicalMaxwellian2D` exists. It is written in terms of the canonical toroidal momentum, but it is more work to set up.

Your markers don't represent the whole EP distribution. They only track how far the EPs have moved away from a reference state. That reference is the background, f0:

f = f0 + δf
    f0 = the reference, written as an exact formula (no noise)
    δf = the difference, carried by the markers
An analogy is ocean waves measured from mean sea level. Yight of the water, only how far each point sits above orbelow sea level. That works well as long as sea level itself isn't moving.

The three things you can set

1. Background, add_background(f0): required. This is "sea level", the state the EPs would stay in with no wave present. The code subtracts it
   out and only simulates the difference.
2. Initial condition, add_initial_condition(f_init): optional. This is what the EPs actually look like at t = 0. If you leave it out, it equals
   f0, so δf starts at zero.
3. Perturbation: a small kick inside the initial condition. You would add one to start the EPs slightly away from f0. For FEEC fields you used
   add_perturbation. For particles, the kick goes inside e.g. n = (profile, TorusModesCos(...)) means "densityprofile plus a small ripple".

What to do for your TAE

Do what you already do in LinearMHD: kick the MHD velocity with your TorusModesSin/Cos, and leave the EPs at f0. The wave then disturbs the EPs
by itself, and their response (δf) builds up naturally. T you want to measure, so you don't need to perturb theEPs directly.

The one warning

f0 must be something that wouldn't change on its own, i.e. an equilibrium. Otherwise "sea level" drifts: δf grows even without a wave, and it can look like a fake drive.

- A uniform Maxwellian with B0=equil.absB0 is a perfect e

---
## 4  Markers: `set_markers(...)`

```python
model.energetic_ions.set_markers(
    loading_params  = LoadingParameters(...),   # how many markers, where, which velocities
    weights_params  = WeightsParameters(...),   # full-f or delta-f (control variate)
    boundary_params = BoundaryParameters(...),  # what happens at eta1 = 0, 1 (and eta2, eta3)
    sorting_params  = SortingParameters(...),   # memory sorting for speed (optional)
    saving_params   = SavingParameters(...),    # orbits and binned f to save
)
```

### 4.1  `LoadingParameters`
| field | default | what it does / advice |
|---|---|---|
| `Np` | 5000 | total number of markers. Alternatives: `ppc` (per grid cell) or `ppb` (per sorting box) |
| `seed` | None | set it for reproducible runs and fair comparisons |
| `loading` | `"pseudo_random"` | also `"sobol_standard"`, `"sobol_antithetic"` (quasi-random: lower noise), `"restart"`, `"external"`, `"tesselation"` |
| **`moments`** | `None` → `(0, 0, 1, 1)` | **the sampling distribution** $s$: `(u_par, 0.0, vth_par, vth_perp)`. It is **not** taken from your background automatically |
| **`B0`** | 2.0 | the |B| used to sample μ (`exp(-mu*B0/vth_perp**2)`). Set it to the typical |B| (3.0 here) |
| `spatial` | `"uniform"` | uniform in η, or `"disc"` (uniform in area of a disc) |
| `specific_markers` | None | tuple of `(eta1, eta2, eta3, v_par, mu)`. It overwrites the first markers. Ideal for orbit studies (Section 6) |

Markers are drawn from $s$, and each carries the weight $w = f_{\rm init}/(s\,N_p)$. **Any $s$ is correct in principle,
but a bad $s$ gives wildly varying weights, and therefore noise.** The next experiment makes this concrete.

We first need a small, fast playground: a periodic slab, a homogeneous field along $z$, and a tiny grid.

4.1: LoadingParameters, i.e. where you put the markers

The core idea: markers are like people in an opinion poll

You can't follow all ~10^19 real EPs, so you follow a sample of Np markers, the way a pollster asks 1000 people instead of the whole country.

Pollsters often oversample a group, for example asking exe enough data on them. They then down-weight each of those answers so the total is still fair. Markers work the same way:
- You choose where to put markers. This is the sampling d
- Each marker gets a weight that corrects for your choice:

w = f / (s · Np)     "how many real particles this marker stands for"
  marker in a region you oversampled   → small weight
  marker in a region you undersampled  → large weight                                                                                        
With the weights, the answer is correct in principle for any choice of s. The choice only affects how noisy the result is.                   
Why a bad s causes noise                                                                                                                     
Suppose you place markers where there are few real particles, and hardly any where most of them are. The few markers in the important region then get huge weights. The result depends on a handful ofthree people speak for half the country, so it jumpsaround randomly.

In [ ]:
ALL_PROPS = ("PushGuidingCenterBxEstar", "PushGuidingCenterParallel", "ShearAlfvenCurrentCoupling5D",
             "Magnetosonic", "CurrentCoupling5DDensity", "CurrentCoupling5DGradB", "CurrentCoupling5DCurlb")

def default_propagator_options(model, pusher_algo="explicit"):
    '''Assign Options to every existing propagator (respects turn_off). Explicit RK4 pushers: see Section 5.3.'''
    P = model.propagators
    for name, prop in vars(P).items():
        prop.options = prop.Options()
    if hasattr(P, "push_bxe"):
        P.push_bxe.options = P.push_bxe.Options(algo=pusher_algo)
    if hasattr(P, "push_parallel"):
        P.push_parallel.options = P.push_parallel.Options(algo=pusher_algo)

def make_slab_sim(folder, background, init=None, Np=20_000, moments=None, control_variate=True,
                  n_markers=4, binning_plots=(), dt=0.05, Tend=0.5, turn_off=(None,), seed=1234,
                  velocity_perturbation=None, save_step=1):
    '''Tiny periodic slab (B = e_z, |B|=1) with LinearMHDDriftkineticCC. Returns (model, sim).'''
    model = LinearMHDDriftkineticCC(turn_off=turn_off)
    for sp in (model.em_fields, model.mhd):
        for var in sp.variables.values():
            var.save_data = True
    model.energetic_ions.var.save_data = True

    sim = Simulation(
        model=model,
        env=EnvironmentOptions(out_folders=RUNS, sim_folder=folder, save_step=save_step),
        time_opts=Time(dt=dt, Tend=Tend),
        domain=domains.Cuboid(r1=1.0, r2=1.0, r3=10.0),
        equil=equils.HomogenSlab(B0z=1.0, beta=0.1),
        grid=grids.TensorProductGrid(num_elements=(4, 4, 16)),
        derham_opts=DerhamOptions(degree=(1, 1, 3)),
    )
    model.energetic_ions.set_markers(
        loading_params=LoadingParameters(Np=Np, seed=seed, B0=1.0, moments=moments),
        weights_params=WeightsParameters(control_variate=control_variate),
        boundary_params=BoundaryParameters(bc=("periodic", "periodic", "periodic")),
        saving_params=SavingParameters(n_markers=n_markers, binning_plots=binning_plots),
    )
    default_propagator_options(model)
    model.energetic_ions.var.add_background(background)
    if init is not None:
        model.energetic_ions.var.add_initial_condition(init)
    if velocity_perturbation is not None:
        model.mhd.velocity.add_perturbation(velocity_perturbation)
    return model, sim

def ess(w):
    '''Effective sample size: how many "equal-weight" markers the weights are worth.'''
    return w.sum() ** 2 / (w**2).sum()

def load_scalars(path_out):
    '''All scalars saved by a run, plus time, as a dict of numpy arrays (no post-processing needed).'''
    with h5py.File(os.path.join(path_out, "data", "data_proc0.hdf5"), "r") as f:
        out = {k: f["scalar"][k][:] for k in f["scalar"]}
        out["t"] = f["time/value"][:]
    return out

### 4.2  Experiment: importance sampling
The target is a **beam**: $u_\parallel=1.5$, $v_{th\parallel}=0.3$. We load it (a) with the default sampling moments and (b) with matched moments,
then compare the weights and the recovered $v_\parallel$ distribution. `sim.allocate()` builds everything
(grid, markers, weights) **without running**, which is very useful for inspecting a set-up.

In [ ]:
beam_bg = maxwellians.GyroMaxwellian2D(n=(1.0, None), u_para=(1.5, None), vth_para=(0.3, None), vth_perp=(0.5, None), B0=1.0)

results = {}
for label, moments in [("default moments (0,0,1,1)", None), ("matched moments (1.5,0,0.3,0.5)", (1.5, 0.0, 0.3, 0.5))]:
    model, sim = make_slab_sim(f"sampling_{'default' if moments is None else 'matched'}", beam_bg,
                               moments=moments, control_variate=False)
    sim.allocate()
    P = model.energetic_ions.var.particles
    w = P.weights.copy()
    results[label] = (P, w)
    print(f"{label:34s}  Np={P.Np}, ESS={ess(w):8.0f} ({100*ess(w)/P.Np:5.1f}% of Np),  max w / mean w = {w.max()/w.mean():6.1f}")

fig, axs = plt.subplots(1, 2, figsize=(12, 3.6), layout="constrained")
for i, (label, (P, w)) in enumerate(results.items()):
    axs[0].hist(w / w.mean(), bins=np.logspace(-4, 2, 80), histtype="step", lw=1.8, color=C[i], label=label)
    edges = np.linspace(0, 3, 61)
    f_slice, _ = P.binning([False, False, False, True, False], [edges])
    axs[1].plot(0.5 * (edges[1:] + edges[:-1]), f_slice, color=C[i], label=label)
axs[0].set(xscale="log", yscale="log", xlabel="w / <w>", ylabel="count", title="weight distribution")
vv_ = np.linspace(0, 3, 300)
axs[1].plot(vv_, np.exp(-(vv_ - 1.5)**2 / (2 * 0.3**2)) / np.sqrt(2 * np.pi) / 0.3, "k:", lw=1.2, label="exact")
axs[1].set(xlabel=r"$v_\parallel$", ylabel=r"binned $f(v_\parallel)$", title="recovered distribution (same Np)")
axs[0].legend(fontsize=8); axs[1].legend(fontsize=8); plt.show()

The experiment in §4.2 loads the same beam (particles moving at v_par ≈ 1.5 with a narrow spread of 0.3) twice, with 20,000 markers each time:
- Blue: markers placed with the default sampling (a Maxwellian centred on 0 with vth = 1). That is the wrong place, since the beam is at 1.5.
- Orange: markers placed with matched sampling, i.e. exactly where the beam is.

It is the poll analogy from before: the same number of people asked, but in the wrong group versus the right one.

In a simulation this noise doesn't just make a plot look rough. The markers deposit their current onto the grid every time step, and that current pushes on the MHD fluid (the coupling term from Q1). Noisy markers mean a noisy force on the wave, which can hide or fake the small TAE growth you are trying to measure.

The fix costs nothing: match moments to your background. For your EPs that means moments=(0, 0, 2.84, 2.84) and B0=3.0. You get the orange behaviour for free with the same Np.

With the default moments, most markers sit where the beam is not ($|v_\parallel|<1$) and carry almost zero weight.
The few markers in the beam carry huge weights, so the ESS collapses and the binned $f$ is noisy.
**Rule: set `moments=(u_par, 0.0, vth_par, vth_perp)` to match, or slightly exceed, the width of what you load.**
For the TAE case (a Maxwellian with $v_{th}$ ≈ 2–3), set `moments=(0.0, 0.0, vth, vth)` and `B0=3.0`.

**Checking what you loaded.** `P.binning(components, bin_edges)` returns the binned `(f, δf)` for any 1D or 2D slice
(the `components` mask selects among $(\eta_1,\eta_2,\eta_3,v_\parallel,\mu)$). `P.show_distribution_function(...)` also plots an analytic reference and returns the max relative error.
It is reliable for the η and $v_\parallel$ slices. **For the μ slice, its analytic reference is wrong** (it is off by a large factor even when the markers are right),
so compare μ against the exact marginal $\frac{B_0}{v_{th\perp}^2}e^{-\mu B_0/v_{th\perp}^2}$ yourself:

In [ ]:
P, _ = results["matched moments (1.5,0,0.3,0.5)"]
# show_distribution_function calls plt.show() itself, so set up the figure and title *before* calling it
plt.figure(figsize=(6, 3.3)); plt.title("built-in check, $v_\\parallel$ slice")
err = P.show_distribution_function([False, False, False, True, False], [np.linspace(0.3, 2.7, 41)], do_plot=True)
print("v_par slice, max rel. error:", err)

edges = np.linspace(0.0, 1.0, 41); mc = 0.5 * (edges[1:] + edges[:-1])
f_mu, _ = P.binning([False, False, False, False, True], [edges])
B0, vperp = 1.0, 0.5
fig, ax = plt.subplots(figsize=(6, 3.3), layout="constrained")
ax.plot(mc, f_mu, color=C[0], label="binned markers")
ax.plot(mc, B0 / vperp**2 * np.exp(-mc * B0 / vperp**2), "k:", label=r"exact $\frac{B_0}{v_{th\perp}^2}e^{-\mu B_0/v_{th\perp}^2}$")
ax.set(xlabel=r"$\mu$", ylabel=r"$f(\mu)$", title=r"$\mu$ slice vs exact marginal"); ax.legend(); plt.show()

- Plot 1, the v_par slice: the markers binned in v_par (blue) against the exact beam (red dashed). They agree, with a maximum error of about 5%, which is normal noise for 20,000 markers.
- Plot 2, the μ slice: the markers binned in μ against the exact curve. It is a decaying exponential (more particles at small μ, as in 3.1), and they agree. This one is checked by hand because Struphy's built-in μ reference is wrong, as the markdown note says.

### 4.3  `WeightsParameters`: full-f vs δf (control variate)
| field | default | meaning |
|---|---|---|
| `control_variate` | False | if True, markers carry $\delta w=(f-f_0)/s/N_p$. Only the deviation from the background is noisy |
| `reject_weights`, `threshold` | False, 0.0 | drop markers with $w_0<$ threshold (saves memory in the tails) |

For linear TAE physics, where $\delta f/f_0\sim10^{-3}$, **δf is essential**. With full-f, the statistical noise $\sim f_0/\sqrt{N_p}$ swamps the signal.
The scalars `en_fv` and `en_fB` then measure the **change** in EP energy relative to $f_0$, which is exactly the quantity that goes into or comes out of the wave.

In [ ]:
pert = perturbations.ModesCos(ls=(0,), ms=(0,), ns=(1,), amps=(0.05,))     # 5% density modulation along z (logical eta3)
bg  = maxwellians.GyroMaxwellian2D(n=(1.0, None), vth_para=(1.0, None), vth_perp=(1.0, None), B0=1.0)
ini = maxwellians.GyroMaxwellian2D(n=(1.0, pert), vth_para=(1.0, None), vth_perp=(1.0, None), B0=1.0)

fig, axs = plt.subplots(1, 2, figsize=(12, 3.6), layout="constrained")
edges = np.linspace(0, 1, 33)
for i, cv in enumerate([False, True]):
    model, sim = make_slab_sim(f"cv_{cv}", bg, init=ini, control_variate=cv, moments=(0.0, 0.0, 1.0, 1.0))
    sim.allocate()
    P = model.energetic_ions.var.particles
    w = P.weights
    print(f"control_variate={cv}:  <|w|>={np.abs(w).mean():.2e},  std(w)={w.std():.2e},  w0 (initial full-f weight) mean={P.markers[~P.holes, P.index['w0']].mean():.2e}")
    axs[0].hist(w * P.Np, bins=100, histtype="step", lw=1.8, color=C[i], label=f"control_variate={cv}")
    f_slice, df_slice = P.binning([False, False, True, False, False], [edges])
    axs[1].plot(0.5 * (edges[1:] + edges[:-1]), df_slice if cv else f_slice - 1.0, color=C[i],
                label="binned δf (CV)" if cv else "binned f - 1 (full-f)")
ee = np.linspace(0, 1, 200)
axs[1].plot(ee, 0.05 * np.cos(2 * np.pi * ee), "k:", lw=1.2, label="exact δn")
axs[0].set(xlabel=r"$w\,N_p$", ylabel="count", title="weights"); axs[0].legend()
axs[1].set(xlabel=r"$\eta_3$", ylabel=r"$\delta n$", title="density perturbation recovered from the same markers"); axs[1].legend(fontsize=8)
plt.show()

Left plot: the weights

- Blue (full-f): all weights sit around 10, between about 9.5 and 10.5. Every marker carries the whole background (about 10, which is the box volume 1×1×10) plus or minus the 5% ripple. Almost all of each weight is the boring background.
- Orange (δf): weights between about −0.5 and +0.5, centred on 0. The background has been removed, so each marker carries only its share of the ripple. Weights can be negative, meaning "fewer particles than the background here".

Both histograms have spikes at the edges. This is simply the shape of a cosine: markers are spread evenly in z, and cos(2πz) spends most of its time near +1 and −1, passing through the middle quickly.

The printed numbers say the same thing:

full-f:  mean |w| = 5.0e-4   spread = 1.8e-5   → spread is 3.5% of the weight
δf:      mean |w| = 1.6e-5   spread = 1.8e-5   → the wei
The spread (the ripple) is the same in both cases. δf just removes the large constant part that every marker was carrying.

Right plot: recovering the ripple from the markers

The black dotted line is the exact 5% cosine.

- Orange (δf): sits almost exactly on the cosine.
- Blue (full-f): the right trend is buried in noise, jum.e. errors twice as big as the signal.

Why the difference is so large: each bin holds about 20,andom counting noise is about 1/sqrt(600) ≈ 4%.
- In full-f, that 4% noise applies to the whole density (1.0), giving about ±0.04, the same size as the 5% signal. The signal is lost.
- In δf, the noise applies only to the ripple (0.05), gies smaller.

### 4.4  `BoundaryParameters`, `SortingParameters`, `SavingParameters`

**`BoundaryParameters(bc=(bc_eta1, bc_eta2, bc_eta3))`**, where each entry is `"periodic" | "reflect" | "remove" | "refill"`.
* Torus: `bc=("remove", "periodic", "periodic")`. A marker crossing $\eta_1=0$ (the inner hole of `HollowTorus`) or $\eta_1=1$ (the wall) is lost and counted in `n_lost_particles`. `"reflect"` keeps the markers but is unphysical at the wall. Use it only to test.
* `bc_sph` and `mean_velocity_index` are for SPH models. Ignore them here.

**`SortingParameters(do_sort, sorting_frequency, boxes_per_dim, box_bufsize, dims_mask)`**
Sorts markers in memory by spatial box for cache efficiency. It is optional and only pays off at large `Np`. `EnvironmentOptions(sort_step=...)` controls how often.

**`SavingParameters(n_markers, binning_plots, kernel_density_plots)`**
* `n_markers`: how many marker orbits to save (the first ids, which are also the ones set by `specific_markers`). After `sim.pproc()`, they are in `sim.orbits.energetic_ions[time, marker, column]` with columns `(x, y, z, v_par, mu, weight, …, id)`, positions **physical**.
* `binning_plots`: a tuple of `BinningPlot(slice, n_bins, ranges, divide_by_jac=True, output_quantity="density")`.
  For `Particles5D`, the coordinate names are `e1, e2, e3` (logical space), **`v1` = $v_\parallel$**, and **`v2` = $\mu$**.
  Examples: `"v1"`, `"e1"`, `"e1_v1"` (radial–parallel velocity, the classic resonance plot), `"v1_v2"`.
  Both `f` and **`δf = f − f0`** are saved at every `save_step`. After `pproc`, they are in `sim.f.energetic_ions.<slice>_<quantity>.f_binned / delta_f_binned / grid_*`.
  `output_quantity` can also be `"current_1..3"` or `"energy_tensor_ij"`.

---
## 5  The seven propagators and their `Options`

Assign options exactly as for `LinearMHD`, one per propagator that exists:
```python
model.propagators.<name>.options = model.propagators.<name>.Options(...)
```
If you forget one, it silently uses the defaults. The next cell prints every default so you can see all the knobs at once.

In [ ]:
m = LinearMHDDriftkineticCC()
for name, prop in vars(m.propagators).items():
    print(f"--- model.propagators.{name}  ({type(prop).__name__})")
    print(prop.Options())

 Each time step is not done in one go. The physics is split into pieces, and each piece is advanced in turn:

LinearMHD:     shear_alf → mag_sonic                          (2 pieces)
this model:    push_bxe → push_parallel → shearalfen_cc5d
               → magnetosonic → cc5d_density → cc5d_gradb → cc5d_curlb   (7 pieces)
It's like a relay race: each propagator runs its own leg, then hands over to the next one. Together they make one full time step.

### 5.1  Reference for each propagator

**`push_bxe`** and **`push_parallel`** (particle pushers; they share the same Options)

| option | default | notes |
|---|---|---|
| `algo` | `"discrete_gradient_1st_order"` | also `"discrete_gradient_2nd_order"`, `"discrete_gradient_1st_order_newton"`, **`"explicit"`** (Runge–Kutta via `butcher`). **See 5.3 before trusting the default in a torus** |
| `butcher` | `None` → `ButcherTableau("rk4")` for explicit | `"forward_euler"`, `"heun2"`, `"rk2"`, `"heun3"`, `"rk4"`, `"3/8 rule"` |
| `maxiter`, `tol` | 20, 1e-7 | fixed-point iteration controls for the discrete-gradient schemes |
| `mpi_sort` | `"each"` | when to send markers to the rank that owns them: after each stage (`"each"`) or only at the end (`"last"`, faster, fine for small dt) |
| `evaluate_e_field` | False | include $-\nabla\phi$. No φ exists in this model, so leave it False |

Discrete-gradient schemes are designed to conserve the guiding-center energy exactly. Explicit RK4 does not, but it is 4th-order accurate and robust.

**`cc5d_gradb`** (∇B-drift current ↔ U, and markers E×B-advected)

| option | default | notes |
|---|---|---|
| `algo` | `"explicit"` (RK4) | or `"discrete_gradient"` with `dg_solver_params=DiscreteGradientSolverParameters(relaxation_factor, tol, maxiter)` |
| `ep_scale` | 1.0 | multiplies the accumulated EP contribution (see 5.2) |
| `solver`, `precond`, `solver_params` | `"pcg"`, `MassMatrixPreconditioner`, `SolverParameters()` | inverts the weighted mass matrix |
| `filter_params` | `FilterParameters()` (off) | smooths the accumulated current (see 5.2) |
| `u_space` | `"Hdiv"` | must match the space of `mhd.velocity` (Hdiv in this model) |

**`cc5d_curlb`** (curvature current ↔ U, markers $v_\parallel$): Crank–Nicolson with a Schur solve. Options are `ep_scale, u_space, solver ("pcg"), precond, solver_params, filter_params`.

**`cc5d_density`** (the $n_{gc}\tilde U\times B/\varepsilon$ term): an implicit linear solve with a *non-symmetric* matrix, hence `solver="pbicgstab"` (or `"bicgstab"`, `"gmres"`). Options are `ep_scale, u_space, precond, solver_params, filter_params`.

**`shearalfen_cc5d`** (shear Alfvén plus EP magnetization; the equivalent of `shear_alf`)

| option | default | notes |
|---|---|---|
| `algo` | `"implicit"` (Crank–Nicolson, Schur complement) | `"explicit"` + `butcher` is also possible (CFL-limited by the Alfvén speed) |
| `solver` / `precond` | `"pcg"` / **`MassMatrixDiagonalPreconditioner`** | note the different default from `shear_alf` |
| `solver_params` | `SolverParameters(tol=1e-8, maxiter=3000, info=False, recycle=True)` | `recycle=True` warm-starts from the previous solution. Loosen `tol` (1e-6) to save time |
| `nonlinear` | **True** | adds the operator built from the *perturbed* $\tilde{\mathbf B}$ (updated every step). For a linear model you can set it to **False** |
| `ep_scale`, `filter_params` | 1.0, off | apply to the μ-magnetization accumulation |

**`magnetosonic`**: identical to `mag_sonic` in `LinearMHD` (`u_space`, `solver="pbicgstab"`, `precond`, `solver_params`).

### 5.2  The coupling knobs: `ep_scale` and `filter_params`
* **`ep_scale`** multiplies what the markers deposit on the grid in that propagator.
  `ep_scale=0.0` in all `cc5d_*` and in `shearalfen_cc5d` makes the EPs **passive** (they feel the wave but do not act back). This is a clean way to separate "EP response" from "EP feedback". Values ≠ 1 break exact energy conservation, which is expected.
  To remove a term completely, including its particle push, use `turn_off` instead (5.4).
* **`FilterParameters(use_filter, modes, repeat, alpha)`** removes particle noise from the accumulated moments before it reaches the MHD:
  * `"fourier_in_tor"`: keeps only the toroidal Fourier indices in `modes` (logical $\eta_3$ indices). For your TAE with `ns=(-1,-1)` on a `tor_period=6` domain, the index is `modes=(1,)`.
    **This requires no MPI decomposition in η3**: `grids.TensorProductGrid(..., mpi_dims_mask=(True, True, False))`, and `num_elements[2] >= 2*max(modes)`.
  * `"three_point"`: a binomial (1-2-1-type) smoother, applied `repeat` times with weight `alpha`, in every direction.
  * `"hybrid"`: three-point first, then Fourier.
  For TAE runs, `FilterParameters(use_filter="fourier_in_tor", modes=(1,))` in all four coupling propagators is a strong noise reducer.

there are really only three kinds of job.

Group 1: move the particles (pushers)

- push_bxe: moves guiding centres by the ∇B drift, the slow sideways drift caused by B being stronger on the inboard side.
- push_parallel: moves them along the field line. This includes the curvature drift and the mirror force that creates trapped bananas.

These only move markers. They don't touch the fluid. The main setting is algo, i.e. which numerical method to use. Use "explicit"; 5.3 shows why.

Group 2: make particles and fluid talk to each other (the

- cc5d_gradb: current from the ∇B drift → pushes the flui
- cc5d_curlb: current from the curvature drift → pushes the fluid U.
- cc5d_density: the "EPs moving with the fluid" correctio
- shearalfen_cc5d: the shear Alfvén wave itself (your TAE lives here) plus the magnetization current from gyration.

Each of these collects the markers' current on the grid, then solves an equation for the new U. That is why they have solver, precond and tol
settings: the same kind of linear-algebra settings you kn

Group 3: pure MHD

- magnetosonic: the compressional part. Identical to mag_sonic in LinearMHD.

### 5.3  Experiment: which pusher algorithm to trust?
We move markers in the ITPA torus with **only `push_bxe`** on, for one step of `dt=0.5`, and look at how far they jump radially.
In `dt=0.5`, the ∇B drift moves a marker by only about $\varepsilon\mu|\nabla B|/B\cdot dt\sim10^{-4}$.

In [ ]:
def pusher_jump_test(algo, which="PushGuidingCenterBxEstar", dt=0.5, Np=20_000):
    turn_off = tuple(p for p in ALL_PROPS if p != which)
    model = LinearMHDDriftkineticCC(turn_off=turn_off)
    sim = Simulation(
        model=model, env=EnvironmentOptions(out_folders=RUNS, sim_folder=f"pusher_{which}_{algo}"),
        time_opts=Time(dt=dt, Tend=dt),
        domain=domains.HollowTorus(a1=0.1, a2=1.0, R0=10.0, sfl=False, pol_period=1, tor_period=6),
        equil=equils.AdhocTorus(a=1.0, R0=10.0, B0=3.0, q_kind=0, p_kind=1, q0=1.71, q1=1.87, p1=0.95, p2=0.05, beta=0.0018),
        grid=grids.TensorProductGrid(num_elements=(8, 32, 4)),
        derham_opts=DerhamOptions(degree=(2, 2, 2), bcs=(("dirichlet", "dirichlet"), None, None)),
    )
    model.energetic_ions.set_markers(loading_params=LoadingParameters(Np=Np, seed=1, B0=3.0),
                                     weights_params=WeightsParameters(control_variate=True),
                                     boundary_params=BoundaryParameters(bc=("remove", "periodic", "periodic")))
    default_propagator_options(model, pusher_algo=algo)
    model.energetic_ions.var.add_background(maxwellians.GyroMaxwellian2D(n=(0.01, None), B0=sim.equil.absB0))
    sim.allocate()
    P = model.energetic_ions.var.particles
    m0 = P.markers[~P.holes].copy()
    getattr(model.propagators, "push_bxe" if "BxE" in which else "push_parallel")(dt)
    m1 = P.markers[~P.holes]
    pos1 = dict(zip(m1[:, -1].astype(int), m1[:, 0]))
    lost = np.array([int(i) not in pos1 for i in m0[:, -1]])
    d_eta1 = np.array([pos1[int(i)] for i in m0[~lost, -1]]) - m0[~lost, 0]
    return lost, np.abs(d_eta1), m0

fig, axs = plt.subplots(1, 2, figsize=(12, 3.6), layout="constrained")
for i, algo in enumerate(["discrete_gradient_1st_order", "explicit"]):
    t0 = time.time()
    lost, jump, m0 = pusher_jump_test(algo)
    print(f"{algo:30s} lost {lost.sum():5d}/{len(lost)} markers in ONE step;  |d eta1|: median {np.median(jump):.1e}, "
          f"99% {np.quantile(jump, .99):.1e}, max {jump.max():.1e}   ({time.time()-t0:.0f}s)")
    axs[0].hist(jump, bins=np.logspace(-8, 0, 80), histtype="step", lw=1.8, color=C[i], label=algo)
    if lost.any():
        axs[1].hist(m0[lost, 0], bins=40, range=(0, 1), histtype="step", lw=1.8, color=C[i], label=f"{algo}: lost markers")
axs[0].set(xscale="log", yscale="log", xlabel=r"$|\Delta\eta_1|$ in one step", ylabel="count", title="push_bxe radial displacement"); axs[0].legend(fontsize=8)
axs[1].set(xlabel=r"initial $\eta_1$ of lost markers", ylabel="count", title="where the lost markers started"); axs[1].legend(fontsize=8)
plt.show()

Load 20,000 markers in your torus, switch on only push_bxe, take one step, and measure how far each marker moved radially (Δη1). It compares two methods:
- discrete_gradient_1st_order (blue): Struphy's default.
- explicit (orange): classic RK4.

Physical expectation: the ∇B drift is slow. In one step a marker should move about 10⁻⁴, or at most around 10⁻³.

Left plot: how far markers moved

- Orange (explicit): a clean hump around 10⁻⁴ that stops sharply by about 4×10⁻³. Every marker moved about as far as the physics says it should.
- Blue (default): the same hump, plus a long tail out to about 0.2. A few percent of markers jumped up to 1000 times too far in a single step. That is physically impossible: a slow drift can't carry a particle a fifth of the way across the plasma in one step. The numerical method is misbehaving for those markers.

Right plot: where the lost markers started

Some of those jumps carry markers out of the domain, where they are removed. 552 were lost in one step with the default method and 0 with explicit. The lost markers started almost all at:
- η1 ≈ 0: the inner hole of HollowTorus.
- η1 ≈ 1: the wall.

So the method breaks down near the edges of the domain. It's not a physical loss: no real particle does that.

**
Losing markers wrongly breaks energy conservation and adds fake EP current. Both can corrupt your TAE growth rate. Raising maxiter/tol doesn't fix it.

Rule: use algo="explicit" for both pushers in the torus, and always watch n_lost_particles. (It's probably worth reporting this to the Struphy developers.)
**

**What this shows, and why it matters.** In this geometry, the default discrete-gradient `push_bxe` produces spurious
radial jumps of up to about 0.2 for a few percent of markers. The lost ones start mostly near the inner boundary $\eta_1\to0$, with some at the wall $\eta_1\to1$.
Those markers are "lost" in a single step, and the energy balance breaks. Explicit RK4 gives displacements consistent
with the drift estimate and loses nothing. Raising `maxiter` and `tol` does not fix it (I checked).

**Recommendation:** use `algo="explicit"` for `push_bxe` (and for `push_parallel`, for consistency) in the torus, and always
watch `n_lost_particles`. A few markers lost from `push_parallel` near $\eta_1=1$ are physical: fast EPs born close to the wall whose orbit width reaches it.
It is worth reporting the discrete-gradient behavior to the Struphy developers.

### 5.4  `turn_off`: switching physics on and off
`turn_off` takes **class names**. Turned-off propagators do not exist at all (no `.options`), which is why
`default_propagator_options` uses `hasattr`. Typical uses:
* pushers only (`turn_off` = all five coupling or MHD propagators): **orbit studies**, no MHD cost (Section 6)
* no `cc5d_*` and `ep_scale=0` in `shearalfen_cc5d`: MHD with test particles
* removing one coupling term at a time: find which one causes a problem

In [ ]:
m = LinearMHDDriftkineticCC(turn_off=("ShearAlfvenCurrentCoupling5D", "Magnetosonic", "CurrentCoupling5DDensity",
                                      "CurrentCoupling5DGradB", "CurrentCoupling5DCurlb"))
print("pushers-only model has propagators:", list(vars(m.propagators)))

---
## 6  Experiment: guiding-center orbits in the ITPA torus
We place four markers at $r=0.5$ on the outboard midplane with `specific_markers`, turn off everything except the two
pushers, and save their orbits (`n_markers=4`). The trapping condition at the outboard midplane is roughly
$|v_\parallel|/v_\perp<\sqrt{2r/R}\approx0.32$, with $v_\perp=\sqrt{2\mu|B|}$.

| marker | $v_\parallel$ | $\mu$ | $v_\parallel/v_\perp$ | expected |
|---|---|---|---|---|
| 0 | +1.5 | 0.05 | 2.7 | co-passing |
| 1 | −1.5 | 0.05 | −2.7 | counter-passing |
| 2 | +0.2 | 0.5 | 0.12 | **trapped (banana)** |
| 3 | +0.6 | 0.5 | 0.35 | barely passing |

About 600 steps; takes about 2 minutes.

In [ ]:
turn_off = ("ShearAlfvenCurrentCoupling5D", "Magnetosonic", "CurrentCoupling5DDensity", "CurrentCoupling5DGradB", "CurrentCoupling5DCurlb")
orb_model = LinearMHDDriftkineticCC(turn_off=turn_off)
orb_model.energetic_ions.var.save_data = True

orb_equil = equils.AdhocTorus(a=1.0, R0=10.0, B0=3.0, q_kind=0, p_kind=1, q0=1.71, q1=1.87, p1=0.95, p2=0.05, beta=0.0018)
orb_sim = Simulation(
    model=orb_model,
    env=EnvironmentOptions(out_folders=RUNS, sim_folder="orbits_torus", save_step=2),
    time_opts=Time(dt=1.0, Tend=600.0),
    domain=domains.HollowTorus(a1=0.1, a2=1.0, R0=10.0, sfl=False, pol_period=1, tor_period=6),
    equil=orb_equil,
    grid=grids.TensorProductGrid(num_elements=(8, 32, 4)),
    derham_opts=DerhamOptions(degree=(2, 2, 2), bcs=(("dirichlet", "dirichlet"), None, None)),
)
e1_05 = (0.5 - 0.1) / 0.9
spec = ((e1_05, 0.0, 0.0,  1.5, 0.05),
        (e1_05, 0.0, 0.0, -1.5, 0.05),
        (e1_05, 0.0, 0.0,  0.2, 0.50),
        (e1_05, 0.0, 0.0,  0.6, 0.50))
orb_model.energetic_ions.set_markers(
    loading_params=LoadingParameters(Np=100, seed=1, B0=3.0, specific_markers=spec),
    boundary_params=BoundaryParameters(bc=("remove", "periodic", "periodic")),
    saving_params=SavingParameters(n_markers=len(spec)),
)
default_propagator_options(orb_model, pusher_algo="explicit")
orb_model.energetic_ions.var.add_background(maxwellians.GyroMaxwellian2D(n=(0.01, None), B0=orb_equil.absB0))

t0 = time.time(); orb_sim.run(); print(f"run: {time.time()-t0:.0f}s")
orb_sim.pproc(); orb_sim.load_plotting_data()
orbits = orb_sim.orbits.energetic_ions          # (time, marker, [x, y, z, v_par, mu, w, ..., id])
print("orbits array:", orbits.shape)

In [ ]:
labels = ["co-passing", "counter-passing", "trapped", "barely passing"]
t_orb = orb_sim.t_grid[: orbits.shape[0]]
R = np.sqrt(orbits[:, :, 0] ** 2 + orbits[:, :, 1] ** 2)
Z = orbits[:, :, 2]

fig, axs = plt.subplots(1, 3, figsize=(15, 4.3), layout="constrained")
th = np.linspace(0, 2 * np.pi, 200)
axs[0].plot(10 + 0.5 * np.cos(th), 0.5 * np.sin(th), color="0.6", lw=1, ls=":", label="r = 0.5 flux surface")
for i in range(4):
    axs[0].plot(R[:, i], Z[:, i], color=C[i], label=labels[i])
axs[0].set(aspect="equal", xlabel="R", ylabel="Z", title="poloidal projection of guiding-center orbits"); axs[0].legend(fontsize=8)
for i in range(4):
    axs[1].plot(t_orb, orbits[:, i, 3], color=C[i], label=labels[i])
axs[1].set(xlabel="t", ylabel=r"$v_\parallel$", title="mirror force: trapped marker bounces"); axs[1].legend(fontsize=8)
for i in range(4):
    axs[2].plot(t_orb, orbits[:, i, 4] - orbits[0, i, 4], color=C[i], label=labels[i])
axs[2].set(xlabel="t", ylabel=r"$\mu(t)-\mu(0)$", title=r"$\mu$ is exactly conserved"); axs[2].legend(fontsize=8)
plt.show()

**Things to notice:**
* The co- and counter-passing orbits are **shifted in opposite directions** from the $r=0.5$ surface. This is the drift-orbit shift, $\propto\varepsilon q v_\parallel$.
* The trapped marker traces a **banana** on the outboard side. Its width is about $\varepsilon q v_\perp/(B\sqrt{r/R})$, of order 0.1 here. $v_\parallel$ changes sign at the bounce points.
* $\mu$ is constant to machine precision, as promised.
* **Try:** change `hot_epsilon` in the constructor (for example 0.05) and watch the orbit widths scale. Or change `pusher_algo` to `"discrete_gradient_1st_order"`.

### 6.1  Experiment: is your EP background an equilibrium?
With δf, a marker's weight changes only if $f_0$ is **not** constant along its orbit. If $f_0$ is not an equilibrium of the unperturbed guiding-center motion,
δf grows secularly **without any wave**. Through the current coupling it then acts as a spurious source on the MHD that can swamp your TAE seed.

* $f_0=n\,\text{Maxwellian}$ with **uniform $n$ and `B0=equil.absB0`** is $\propto e^{-E/T}$ with $E=v_\parallel^2/2+\mu|B|$, which is a function of the conserved energy only. **It is an exact equilibrium.**
* A radial profile $n(r)$ is not ($r$ is not a constant of motion: orbits drift across surfaces).
* The default `B0=2.0` breaks it even for uniform $n$, because the exponent is no longer the energy.

The test runs 20 pusher-only steps and tracks the δf energy `en_fv + en_fB` (about 20 s per case):

In [ ]:
def equilibrium_test(n_moment, B0, label, vth=2.84, Np=20_000, nsteps=20, dt=0.5):
    turn_off = ("ShearAlfvenCurrentCoupling5D", "Magnetosonic", "CurrentCoupling5DDensity", "CurrentCoupling5DGradB", "CurrentCoupling5DCurlb")
    model = LinearMHDDriftkineticCC(turn_off=turn_off)
    equil = equils.AdhocTorus(a=1.0, R0=10.0, B0=3.0, q_kind=0, p_kind=1, q0=1.71, q1=1.87, p1=0.95, p2=0.05, beta=0.0018)
    sim = Simulation(model=model, env=EnvironmentOptions(out_folders=RUNS, sim_folder=f"equilibrium_{label}"),
                     time_opts=Time(dt=dt, Tend=nsteps * dt),
                     domain=domains.HollowTorus(a1=0.1, a2=1.0, R0=10.0, sfl=False, pol_period=1, tor_period=6), equil=equil,
                     grid=grids.TensorProductGrid(num_elements=(8, 32, 4)),
                     derham_opts=DerhamOptions(degree=(2, 2, 2), bcs=(("dirichlet", "dirichlet"), None, None)))
    model.energetic_ions.set_markers(loading_params=LoadingParameters(Np=Np, seed=1, B0=3.0, moments=(0.0, 0.0, vth, vth)),
                                     weights_params=WeightsParameters(control_variate=True),
                                     boundary_params=BoundaryParameters(bc=("remove", "periodic", "periodic")))
    default_propagator_options(model, pusher_algo="explicit")
    B0_arg = equil.absB0 if B0 == "absB0" else B0
    model.energetic_ions.var.add_background(maxwellians.GyroMaxwellian2D(
        n=(n_moment, None), vth_para=(vth, None), vth_perp=(vth, None), B0=B0_arg))
    sim.run()
    return load_scalars(sim.env.path_out)

cases = [("uniform n, B0=equil.absB0", 0.0072, "absB0"),
         ("tanh n(r), B0=equil.absB0", n_ep, "absB0"),
         ("uniform n, B0=2.0 (default)", 0.0072, 2.0)]
fig, ax = plt.subplots(figsize=(8, 3.8), layout="constrained")
for i, (label, n_m, B0) in enumerate(cases):
    t0 = time.time()
    S_eq = equilibrium_test(n_m, B0, label.split(",")[0].replace(" ", "_") + f"_{i}")
    ax.plot(S_eq["t"], S_eq["en_fv"] + S_eq["en_fB"], color=C[i], label=label)
    print(f"{label:30s} δf energy at end: {S_eq['en_fv'][-1] + S_eq['en_fB'][-1]: .2e},  lost markers: {int(S_eq['n_lost_particles'][-1])}   ({time.time()-t0:.0f}s)")
ax.axhline(0, color="0.5", lw=1)
ax.set(xlabel="t", ylabel="en_fv + en_fB", title="δf energy with NO wave: pushers only, ITPA torus"); ax.legend(fontsize=8)
plt.show()

**Reading the result.** The first case fluctuates around zero: that is marker noise, including markers lost at the wall.
The other two grow steadily. That growth is spurious δf, and in the full model it becomes a spurious source for the MHD.

**Why this matters for the TAE run.** You *need* a gradient in $n_{EP}(r)$ to drive the TAE, so you cannot simply use a uniform $n$. In a one-step coupled test
(20k markers, `dt=0.5`, no filter), the tanh-profile background pumped the wave energy from 2.2e-8 (the seed) to 7e-6 in two steps.
With a uniform $n$ plus the Fourier filter, it stayed at about 6e-8. What to do:
1. Keep **`B0=equil.absB0`** (never the default).
2. Use the **toroidal Fourier filter**. The spurious δf from an axisymmetric $f_0$ is mostly $n=0$, and the filter removes it from the coupling. What gets through is marker noise in the kept harmonic.
3. The model is **linear**, so the absolute amplitude is arbitrary. Use a **larger seed** (`amps=1e-2` instead of 1e-3) so the TAE starts well above the spurious level.
4. Use **enough markers** and compare with an `NEP_AXIS=0` run.

Measured in the coupled model (tanh profile plus Fourier filter, `dt=0.5`, 2 steps; wave energy = `en_U + en_B`):

| markers | seed `amps` | wave energy: seed → after 2 steps |
|---|---|---|
| 2e4 | 1e-3 | 2.2e-8 → 6.7e-7 (30×: noise dominates) |
| 2e4 | 1e-2 | 2.2e-6 → 2.9e-6 (the same ≈6.6e-7 added, now small) |
| 2e5 | 1e-3 | 2.2e-8 → 7.6e-8 (the added part drops about 10×, i.e. ∝ $1/N_p$) |
5. A true equilibrium with a gradient must be a function of the constants of motion: `maxwellians.CanonicalMaxwellian2D(n=(n(ψ_c), None), vth=..., equil=equil, epsilon=ε)` (density as a function of the canonical toroidal momentum $\psi_c$).
   It runs in this model, but in my quick pusher-only test it did **not** clearly reduce the δf growth. I did not validate its conventions, so treat it as something to investigate, not a drop-in fix.

---
## 7  Sandbox: the fully coupled model in a slab, from run to post-processing
All seven propagators are on. The set-up has an MHD shear-Alfvén wave along $z$ (`ModesSin` in `mhd.velocity`, as in your LinearMHD
runs), δf markers, and a **5 % EP density perturbation along $z$** in the initial condition (so that δf ≠ 0 from the start).
The goal is to exercise the whole workflow: **run → scalars → energy conservation → `pproc` → binned f/δf → orbits**. It takes about a minute.

In [ ]:
sb_bg   = maxwellians.GyroMaxwellian2D(n=(0.05, None), vth_para=(1.0, None), vth_perp=(1.0, None), B0=1.0)
sb_init = maxwellians.GyroMaxwellian2D(n=(0.05, perturbations.ModesCos(ls=(0,), ms=(0,), ns=(1,), amps=(0.0025,))),
                                       vth_para=(1.0, None), vth_perp=(1.0, None), B0=1.0)
sb_model, sb_sim = make_slab_sim(
    "sandbox_coupled", sb_bg, init=sb_init, Np=20_000, moments=(0.0, 0.0, 1.0, 1.0), control_variate=True,
    n_markers=4, dt=0.1, Tend=6.0, save_step=2,
    binning_plots=(BinningPlot(slice="v1", n_bins=64, ranges=(-4.0, 4.0)),
                   BinningPlot(slice="e3_v1", n_bins=(32, 48), ranges=((0.0, 1.0), (-3.0, 3.0)))),
    velocity_perturbation=perturbations.ModesSin(ls=(0,), ms=(0,), ns=(1,), amps=(1e-3,), comp=0, given_in_basis="v"),
)
# print what will actually be used: worth doing before every production run
for name, prop in vars(sb_model.propagators).items():
    print(f"{name:16s}", prop.options)

In [ ]:
t0 = time.time(); sb_sim.run(); print(f"run: {time.time()-t0:.0f}s")

S = load_scalars(sb_sim.env.path_out)
fig, axs = plt.subplots(1, 2, figsize=(12, 3.6), layout="constrained")
for i, k in enumerate(["en_U", "en_B", "en_p"]):
    axs[0].plot(S["t"], S[k], color=C[i], label=k)
axs[0].set(xlabel="t", ylabel="energy", title="MHD energies"); axs[0].legend()
axs[1].plot(S["t"], (S["en_tot"] - S["en_tot"][0]) / S["en_tot"][0], color=C[0], label="(en_tot - en_tot(0)) / en_tot(0)")
axs[1].set(xlabel="t", ylabel="relative error", title="total energy conservation"); axs[1].legend()
plt.show()
print("EP energies (δf): max|en_fv| =", np.abs(S["en_fv"]).max(), ", max|en_fB| =", np.abs(S["en_fB"]).max(),
      ",  lost markers:", S["n_lost_particles"][-1])

As predicted in §2.4, the EP energy change is tiny in the homogeneous slab: there is no curvature, no ∇B, and no gradient to couple through.
The wave simply oscillates between `en_U` and `en_B`. Look at `en_tot` on **every** new set-up:
* Here it wanders by a few percent **of the wave energy**, which is about 1e-7 in absolute terms, while the EP δf energy is about 4e-4. The explicit RK4 pushers conserve the guiding-center energy only approximately (the discrete-gradient ones are designed to be exact, but see §5.3), and a small drift of a large EP energy is visible next to a small wave.
* Without the initial EP density perturbation, `en_fv` and `en_fB` stay **exactly** 0 here: f₀ is an exact equilibrium, so every δf weight stays 0. `en_tot` is then conserved to about 1e-11, which is the solver tolerance.
* Rule of thumb: compare the `en_tot` drift with the energy changes you want to interpret (the wave growth), not with `en_tot` itself.

Now post-process. This is the same `sim.pproc()` / `sim.load_plotting_data()` you use for LinearMHD, and it additionally produces the kinetic data:

In [ ]:
sb_sim.pproc()
sb_sim.load_plotting_data()

print("binned slices:", [k for k in vars(sb_sim.f.energetic_ions)])
s1 = sb_sim.f.energetic_ions.v1_density
s2 = sb_sim.f.energetic_ions.e3_v1_density
print("v1 slice:   f_binned", s1.f_binned.shape, " grid_v1", s1.grid_v1.shape)
print("e3_v1 slice:", {k: getattr(s2, k).shape for k in vars(s2) if not k.startswith('_')} if hasattr(s2, '__dict__') else dir(s2))

fig, axs = plt.subplots(1, 3, figsize=(16, 3.8), layout="constrained")
axs[0].plot(s1.grid_v1, s1.f_binned[0], color=C[0], label="full f, t=0")
axs[0].set(xlabel=r"$v_\parallel$", ylabel="f", title=r"binned full f($v_\parallel$)"); axs[0].legend()
for i, n in enumerate([0, len(sb_sim.t_grid) - 1]):
    d = s2.delta_f_binned[n]; vmax = np.abs(s2.delta_f_binned[0]).max()
    im = axs[1 + i].pcolormesh(s2.grid_e3, s2.grid_v1, d.T, cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto")
    axs[1 + i].set(xlabel=r"$\eta_3$", ylabel=r"$v_\parallel$", title=rf"δf($\eta_3, v_\parallel$) at t={sb_sim.t_grid[n]:.1f}")
fig.colorbar(im, ax=axs[1:])
plt.show()

o = sb_sim.orbits.energetic_ions
print("saved orbits:", o.shape, "  (time, marker, [x, y, z, v_par, mu, weight, ..., id])")

**Phase mixing.** Each marker streams along $z$ at its own $v_\parallel$, so the initially vertical stripes of δf shear into tilted stripes (slope $\propto 1/t$).
The density perturbation $\int\delta f\,dv_\parallel$ decays even though nothing is dissipated, which is the kinetic mechanism behind Landau damping.
In the torus with a gradient-driven TAE, the
$\eta_1$–$v_\parallel$ δf develops coherent structure at the resonances. That is the `e1_v1` binning set up in the template below.

FEEC fields are post-processed exactly as before (`sb_sim.spline_values.mhd.velocity_log`, and so on), so your existing plotting
functions in `pproc_TAE_benchmark.py` work unchanged for the MHD part of an EP run.

---
## 8  Production: the ITPA TAE with energetic ions

The recipe is to **start from your `params_TAE_benchmark.py`** (domain, equilibrium, grid, Derham, and the `mhd.velocity`
perturbation all stay) and add:
1. `LinearMHDDriftkineticCC(...)` with the EP mass and charge numbers.
2. `set_markers(...)` with matched `moments` and `B0`, δf, `remove` at η1, and useful binning.
3. A `GyroMaxwellian2D` background with a radial density profile and `B0=equil.absB0`.
4. Propagator options: explicit pushers, the Fourier filter in the coupling terms, and `mpi_dims_mask=(True, True, False)`.

The next cell **writes** `tae_ep/params_TAE_EP.py`. Submit it the same way as before (`srun -n 32 python params_TAE_EP.py`).
The numbers marked `# CHOOSE` are physics choices. The ITPA-like values are my best recollection of
Könies et al., *Nucl. Fusion* **58** 126027 (2018): n_EP(0) ≈ 1.44e17 m⁻³, n_bulk = 2e19 m⁻³, T_EP = 400 keV.
**Verify them against the paper** before quoting results. Everything else is a numerical recommendation from this notebook.

In [ ]:
os.makedirs("tae_ep", exist_ok=True)
PARAMS_TAE_EP = r"""
# ---------------------------------------------------------------------------------
# ITPA-like TAE with energetic ions: LinearMHDDriftkineticCC
# Generated by Kinetic_Particles.ipynb. MHD part identical to params_TAE_benchmark.py
# ---------------------------------------------------------------------------------
import logging
import numpy as np

from struphy import set_logging_level
set_logging_level(logging.INFO)

from struphy import (BaseUnits, DerhamOptions, EnvironmentOptions, ProfilingOptions, Simulation, Time,
                     domains, equils, grids, perturbations, maxwellians,
                     LoadingParameters, WeightsParameters, BoundaryParameters, SortingParameters,
                     SavingParameters, BinningPlot)
from struphy.models import LinearMHDDriftkineticCC
from struphy.pic.accumulation.filter import FilterParameters
from struphy.linear_algebra.solver import SolverParameters

import os
HERE = os.path.dirname(os.path.abspath(__file__))

# ---------------- run size (override via environment for quick tests) ----------------
NUM_ELEMENTS = tuple(int(x) for x in os.environ.get("TAE_EP_NEL", "24,96,16").split(","))
DEGREE       = tuple(int(x) for x in os.environ.get("TAE_EP_DEG", "3,3,3").split(","))
NP           = float(os.environ.get("TAE_EP_NP", "2e6"))            # markers (total over all ranks)
DT, TEND     = float(os.environ.get("TAE_EP_DT", "0.5")), float(os.environ.get("TAE_EP_TEND", "500."))
SIM_FOLDER   = os.environ.get("TAE_EP_FOLDER", "sim1_EP")

# ---------------- physics choices ----------------
T_EP_keV     = 400.0      # CHOOSE: EP temperature
A_EP, Z_EP   = 1.0, 1     # CHOOSE: EP mass/charge numbers (ITPA: check the paper; your LinearMHD runs used A_b=1)
NEP_AXIS     = 0.0072     # CHOOSE: EP density in units of n_hat=1e20 (your equil has n_bulk(0)~1), i.e. ratio n_EP/n_bulk
R0_GRAD, DELTA, L_GRAD = 0.5, 0.2, 0.3   # CHOOSE: EP profile, steepest gradient at r=R0_GRAD

# ---------------- model ----------------
base_units = BaseUnits()
model = LinearMHDDriftkineticCC(base_units=base_units, hot_mass_number=A_EP, hot_charge_number=Z_EP)
model.em_fields.b_field.save_data = True
model.mhd.density.save_data = True
model.mhd.pressure.save_data = True
model.mhd.velocity.save_data = True
model.energetic_ions.var.save_data = True

# EP thermal speed in units of v_A (hat)
vth_EP = np.sqrt(T_EP_keV * 1e3 * 1.602176634e-19 / (A_EP * 1.67262192369e-27)) / model.units.v

# ---------------- simulation (same as params_TAE_benchmark.py) ----------------
env = EnvironmentOptions(save_step=2, out_folders=HERE, sim_folder=SIM_FOLDER, max_runtime=350)
time_opts = Time(dt=DT, Tend=TEND)
a1 = 0.1
domain = domains.HollowTorus(a1=a1, a2=1.0, R0=10.0, sfl=False, pol_period=1, tor_period=6)
equil = equils.AdhocTorus(a=1.0, R0=10.0, B0=3.0, q_kind=0, p_kind=1, q0=1.71, q1=1.87,
                          p1=0.95, p2=0.05, beta=0.0018)
# no MPI decomposition along eta3 -> required by the toroidal Fourier filter
grid = grids.TensorProductGrid(num_elements=NUM_ELEMENTS, mpi_dims_mask=(True, True, False))
derham_opts = DerhamOptions(degree=DEGREE, bcs=(("dirichlet", "dirichlet"), None, None))

sim = Simulation(model=model, params_path=__file__, env=env, time_opts=time_opts,
                 domain=domain, equil=equil, grid=grid, derham_opts=derham_opts,
                 profiling_opts=ProfilingOptions())

# ---------------- markers ----------------
model.energetic_ions.set_markers(
    loading_params=LoadingParameters(Np=int(NP), seed=1234, B0=3.0,
                                     moments=(0.0, 0.0, float(vth_EP), float(vth_EP))),   # sample like f0
    weights_params=WeightsParameters(control_variate=True),                               # delta-f
    boundary_params=BoundaryParameters(bc=("remove", "periodic", "periodic")),
    sorting_params=SortingParameters(),
    saving_params=SavingParameters(
        n_markers=20,
        binning_plots=(
            BinningPlot(slice="e1_v1", n_bins=(32, 64), ranges=((0.0, 1.0), (-4 * float(vth_EP), 4 * float(vth_EP)))),
            BinningPlot(slice="v1", n_bins=128, ranges=(-4 * float(vth_EP), 4 * float(vth_EP))),
            BinningPlot(slice="e1", n_bins=64, ranges=(0.0, 1.0)),
        ),
    ),
)

# ---------------- propagator options ----------------
filt = FilterParameters(use_filter="fourier_in_tor", modes=(1,))     # keep only the n=-1 (logical) harmonic
P = model.propagators
P.push_bxe.options      = P.push_bxe.Options(algo="explicit")        # see notebook section 5.3
P.push_parallel.options = P.push_parallel.Options(algo="explicit")
P.cc5d_gradb.options    = P.cc5d_gradb.Options(filter_params=filt)
P.cc5d_curlb.options    = P.cc5d_curlb.Options(filter_params=filt)
P.cc5d_density.options  = P.cc5d_density.Options(filter_params=filt)
P.shearalfen_cc5d.options = P.shearalfen_cc5d.Options(filter_params=filt, nonlinear=False)
P.magnetosonic.options  = P.magnetosonic.Options()

# ---------------- MHD initial perturbation (identical to the LinearMHD run) ----------------
# SEED is 10x the LinearMHD run: the model is linear, and a larger seed keeps the TAE above the
# spurious drive from the non-equilibrium n(r) background (notebook section 6.1)
SEED = float(os.environ.get("TAE_EP_SEED", "1e-2"))
ms_radial_1, ms_radial_2 = 10, 11
model.mhd.velocity.add_perturbation(perturbations.TorusModesSin(
    ms=(ms_radial_1, ms_radial_2), ns=(-1, -1), amps=(SEED, SEED),
    pfuns=("exp", "exp"), pfun_params=([0.5, 0.1], [0.5, 0.1]), comp=0, given_in_basis="2"))
model.mhd.velocity.add_perturbation(perturbations.TorusModesCos(
    ms=(ms_radial_1, ms_radial_2), ns=(-1, -1),
    amps=(SEED / (2 * np.pi * ms_radial_1), SEED / (2 * np.pi * ms_radial_2)),
    pfuns=("d_exp", "d_exp"), pfun_params=([0.5, 0.1], [0.5, 0.1]), comp=1, given_in_basis="2"))

# ---------------- EP background (= initial condition) ----------------
def n_ep(*etas):
    e1 = etas[0][:, 0] if len(etas) == 1 else etas[0]
    r = a1 + (1.0 - a1) * e1
    return NEP_AXIS * np.exp(-DELTA / L_GRAD * np.tanh((r - R0_GRAD) / DELTA))

f0 = maxwellians.GyroMaxwellian2D(n=(n_ep, None), vth_para=(float(vth_EP), None), vth_perp=(float(vth_EP), None),
                                  B0=equil.absB0)
model.energetic_ions.var.add_background(f0)

if __name__ == "__main__":
    sim.run()
"""
with open("tae_ep/params_TAE_EP.py", "w") as fh:
    fh.write(PARAMS_TAE_EP.lstrip())
print("written:", os.path.abspath("tae_ep/params_TAE_EP.py"))

**Smoke test.** Import the file with a reduced grid (through the environment variables it reads), check the set-up, and run 2 steps (about 2 min).
Importing does not run the simulation, thanks to `if __name__ == "__main__"`. This is the same trick your `pproc` scripts use.

In [ ]:
RUN_SMOKE_TEST = True

os.environ.update(TAE_EP_NEL="8,32,4", TAE_EP_DEG="2,2,2", TAE_EP_NP="2e4", TAE_EP_DT="0.5", TAE_EP_TEND="1.0",
                  TAE_EP_FOLDER=os.path.join(RUNS, "tae_ep_smoke"))
spec_ = importlib.util.spec_from_file_location("params_TAE_EP", "tae_ep/params_TAE_EP.py")
tae = importlib.util.module_from_spec(spec_); spec_.loader.exec_module(tae)
set_logging_level(logging.WARNING)
print(f"vth_EP = {tae.vth_EP:.3f} v_A_hat,   epsilon = {tae.model.energetic_ions.equation_params.epsilon:.4f}")

if RUN_SMOKE_TEST:
    t0 = time.time(); tae.sim.run(); print(f"2 steps: {time.time()-t0:.0f}s")
    S_smoke = load_scalars(tae.sim.env.path_out)
    for k in ["en_U", "en_B", "en_p", "en_fv", "en_fB", "en_tot", "n_lost_particles"]:
        print(f"{k:18s}", np.array2string(S_smoke[k], precision=3))
for k in ["TAE_EP_NEL", "TAE_EP_DEG", "TAE_EP_NP", "TAE_EP_DT", "TAE_EP_TEND", "TAE_EP_FOLDER"]:
    os.environ.pop(k, None)

**Reading the smoke test.** The test only checks that the file builds and steps. Its numbers are **not physics**:
* 2e4 markers on a coarse grid: the EP δf, and the wave energy it drives, are dominated by noise and by the non-equilibrium background (§6.1).
* `en_tot` changes by several 1e-3, dominated by the EP δf energy. Two sources contribute: the ~80–100 markers lost at the wall in the first steps (`n_lost_particles`, **physical prompt loss** of fast ions born near the edge with orbit widths ~ $q\rho_h$, which carry energy out of the system), and the non-equilibrium n(r) background (§6.1).
* What you want to see in the real run: `n_lost_particles` saturating after the first bounce times, the wave energy growing exponentially (γ > 0) above an `NEP_AXIS=0` control, and δf structure at the resonances.

**Planning the real run** (32 ranks, as in your `submit.sh`, just pointing at the new file):
* **Cost.** `shearalfen_cc5d` dominates: its Schur-complement PCG needs about 100 iterations at `dt=0.5`, and each iteration applies the projection operators.
  Your commit "much faster solver because of shorter timestep" is the same effect: fewer iterations per step at smaller `dt`.
  The pushers and accumulations scale with `Np/ranks`.
* **Markers.** δf noise ∝ $1/\sqrt{N_p}$. Start with 1–4 × 10⁶ markers on 32 ranks and **check convergence of the growth rate in `Np`**.
* **Before a long job**, check in the log: `n_lost_particles` stays small, `en_tot` is conserved, and the PCG iteration counts (`SolverParameters(info=True)`) are sane.
* **`max_runtime`** is in minutes. Restarts work as for LinearMHD (`EnvironmentOptions(restart=True)`), and markers are saved in `restart/`.
* **Passive-EP control run.** Set `ep_scale=0.0` in the four coupling propagators. The TAE should then behave like your LinearMHD run, which is a good consistency check.

---
## 9  Analysis toolkit: energy exchange and growth rates
Energy bookkeeping is the cleanest diagnostic for wave–particle interaction:

$$
\frac{d}{dt}\big(E_U+E_B+E_p\big)=-\frac{d}{dt}\big(E_{fv}+E_{fB}\big)
$$

The EPs drive the wave when their (δf) energy **decreases**. A linear mode has $E_{\rm wave}\propto e^{2\gamma t}$, so fit $\gamma$ from the log slope divided by 2.
Its oscillation (between `en_U` and `en_B`) is at $2\omega$.

The functions below work on any run folder. Here they are applied to your **EP-free baseline** `examples/LinearMHD/itpa_tae_benchmark/sim5_higherResolution`,
which is the reference an EP run must be compared with.

In [ ]:
def growth_rate(t, E, t_min=None, t_max=None):
    '''gamma from E ~ exp(2 gamma t), least-squares fit of log E in [t_min, t_max].'''
    m = np.isfinite(E) & (E > 0)
    if t_min is not None: m &= t >= t_min
    if t_max is not None: m &= t <= t_max
    slope, icpt = np.polyfit(t[m], np.log(E[m]), 1)
    return slope / 2, (slope, icpt, m)

def dominant_frequency(t, E):
    '''Frequency of the wave from the oscillation of E_U (at 2*omega). Returns omega.'''
    x = E - np.polyval(np.polyfit(t, E, 2), t)
    freqs = np.fft.rfftfreq(len(t), t[1] - t[0]) * 2 * np.pi
    spec = np.abs(np.fft.rfft(x * np.hanning(len(x))))
    return freqs[1:][np.argmax(spec[1:])] / 2

def analyze_run(path_out, title, t_fit=(None, None)):
    S = load_scalars(path_out)
    t = S["t"]
    E_wave = S["en_U"] + S["en_B"] + S["en_p"]
    has_ep = "en_fv" in S
    gamma, (slope, icpt, m) = growth_rate(t, E_wave, *t_fit)
    fig, axs = plt.subplots(1, 3 if has_ep else 2, figsize=(16 if has_ep else 11, 3.8), layout="constrained")
    for i, k in enumerate(["en_U", "en_B"]):
        axs[0].plot(t[m], S[k][m], color=C[i], label=k)
    axs[0].plot(t[m], E_wave[m], color=C[2], label="en_U + en_B + en_p")
    axs[0].plot(t[m], np.exp(icpt + slope * t[m]), "k--", lw=1.2, label=f"fit: γ={gamma:.2e}")
    axs[0].set(xlabel="t", title=f"{title}: wave energies (fit window)"); axs[0].legend(fontsize=8)
    axs[1].semilogy(t, np.abs(S["en_tot"]), color=C[0])
    axs[1].axvspan(t[m][0], t[m][-1], color="0.9", zorder=0, label="fit window"); axs[1].legend(fontsize=8)
    axs[1].set(xlabel="t", ylabel="|en_tot|", title="en_tot over the whole run")
    if has_ep:
        axs[2].plot(t, E_wave - E_wave[0], color=C[0], label=r"$\Delta E_{wave}$")
        axs[2].plot(t, -(S["en_fv"] + S["en_fB"] - S["en_fv"][0] - S["en_fB"][0]), "--", color=C[1], label=r"$-\Delta E_{EP}$")
        axs[2].set(xlabel="t", title="energy exchange (curves overlap if conserved)"); axs[2].legend(fontsize=8)
    plt.show()
    print(f"{title}: gamma = {gamma:.3e},  omega ~ {dominant_frequency(t[m], S['en_U'][m]):.4f}  (code units; t_hat = x_hat/v_A_hat)")
    return S

BASELINE = os.path.abspath("../examples/LinearMHD/itpa_tae_benchmark/sim5_higherResolution")
if os.path.exists(os.path.join(BASELINE, "data", "data_proc0.hdf5")):
    S_base = analyze_run(BASELINE, "LinearMHD baseline (sim5)", t_fit=(0, 180))
else:
    print("baseline not found:", BASELINE)

EP_RUN = os.path.abspath("tae_ep/sim1_EP")          # after your cluster run
if os.path.exists(os.path.join(EP_RUN, "data", "data_proc0.hdf5")):
    S_ep = analyze_run(EP_RUN, "with EPs", t_fit=(100, None))

**About the baseline.** Up to $t\approx190$, the energy of sim5 just sloshes between `en_U` and `en_B` (γ ≈ 0, as it should without EPs).
After that, `en_tot` grows exponentially by about six orders of magnitude by $t=230$. This is a **numerical instability of that run, not TAE physics**
(sim3, with the same grid and `dt=0.5`, stays bounded up to $t=136$). Worth investigating before using sim5 as a reference.
Always inspect `en_tot` over the whole run, as in the right panel, and fit only a clean window.

**Kinetic diagnostics for the EP run** (after `sim.pproc()` on the cluster output):
```python
sim = load_sim("tae_ep/sim1_EP")                   # your helper from pproc_TAE_benchmark.py works unchanged
s = sim.f.energetic_ions.e1_v1_density
# s.grid_e1, s.grid_v1, s.delta_f_binned[t_index]  -> pcolormesh(grid_e1, grid_v1, df.T)
```
Look for δf structures at $|v_\parallel|\approx v_A(r)$ and $v_A/3$ near $r\approx0.5$ (convert with $r=0.1+0.9\eta_1$). They are the fingerprint of the resonant drive.
Compare the frequency and $\gamma$ against the LinearMHD baseline and the passive-EP (`ep_scale=0`) run.

---
## 10  Pitfalls checklist

| # | Pitfall | Symptom | Fix |
|---|---|---|---|
| 1 | `GyroMaxwellian2D(B0=2.0)` default | $T_\perp$ off by $|B|/2$ | `B0=equil.absB0` (torus) or the constant |B| (slab) |
| 2 | `LoadingParameters(moments=None)` → `(0,0,1,1)` | noisy, low ESS, a few huge weights | `moments=(u_par, 0, vth_par, vth_perp)` like f0, and `B0≈|B|` |
| 3 | full-f for linear physics | the signal drowns in noise | `WeightsParameters(control_variate=True)` |
| 4 | default discrete-gradient `push_bxe` in the torus | many markers lost at step 1, energy drift | `algo="explicit"` (§5.3) and watch `n_lost_particles` |
| 5 | a callable profile written as `lambda e1, e2, e3: ...` | crash in flat (marker) evaluation | accept `*etas` (use `eta1_profile`) |
| 6 | `add_perturbation` on the kinetic variable | perturbation ignored or an error | put the perturbation in a moment tuple of `add_initial_condition` |
| 7 | `"fourier_in_tor"` filter with MPI decomposition in η3 | assertion error | `TensorProductGrid(..., mpi_dims_mask=(True, True, False))` |
| 8 | resonance mismatch | no drive, EPs look passive | $v_{th}$ comparable to the local $v_A=|B|/\sqrt{n}$ (≈3 here, not 1) |
| 9 | flat EP profile | no drive | put the steepest gradient at the mode location |
| 10 | forgetting an `.options` | silent defaults (e.g. `nonlinear=True`, DG pushers) | loop over `vars(model.propagators)` and `sim.show_propagator_options()` |
| 11 | `turn_off` with the attribute name (`"push_bxe"`) | nothing turned off | use the **class** name (`"PushGuidingCenterBxEstar"`) |
| 12 | `sim.run()` in a module without the `__main__` guard | importing for pproc re-runs | keep `if __name__ == "__main__": sim.run()` |
| 13 | $f_0$ not an equilibrium (n(r) profile, wrong `B0`) | δf and wave energy grow with no physical drive; `en_fv` drifts in a pusher-only run | §6.1: filter, larger seed, more markers, EP-free control run |
| 14 | reading `en_tot` drift as a bug when markers are lost | `en_tot` changes in steps with `n_lost_particles` | lost markers carry energy out; judge conservation with `bc` losses in mind |
| 15 | `show_distribution_function` for the μ slice | reports large errors for correct markers | compare with the exact marginal yourself (§4.2) |

## 11  Cheat sheet: minimal additions to a LinearMHD params file
```python
from struphy import LoadingParameters, WeightsParameters, BoundaryParameters, SavingParameters, BinningPlot, maxwellians
from struphy.models import LinearMHDDriftkineticCC
model = LinearMHDDriftkineticCC(base_units=BaseUnits(), hot_mass_number=1.0, hot_charge_number=1)
model.energetic_ions.var.save_data = True
# ... env, time, domain, equil, grid, derham, sim = Simulation(...) exactly as before ...
model.energetic_ions.set_markers(
    loading_params=LoadingParameters(Np=1e6, seed=1, B0=3.0, moments=(0., 0., vth, vth)),
    weights_params=WeightsParameters(control_variate=True),
    boundary_params=BoundaryParameters(bc=("remove", "periodic", "periodic")),
    saving_params=SavingParameters(n_markers=10, binning_plots=(BinningPlot(slice="e1_v1", n_bins=(32, 64), ranges=((0., 1.), (-4*vth, 4*vth))),)))
for name, p in vars(model.propagators).items():
    p.options = p.Options()
model.propagators.push_bxe.options = model.propagators.push_bxe.Options(algo="explicit")
model.propagators.push_parallel.options = model.propagators.push_parallel.Options(algo="explicit")
model.energetic_ions.var.add_background(maxwellians.GyroMaxwellian2D(n=(n_ep, None), vth_para=(vth, None), vth_perp=(vth, None), B0=equil.absB0))
# MHD perturbation exactly as in LinearMHD
```

## 12  Exercises (each one is a small edit of a cell above)
1. **Orbit width vs ε:** rerun §6 with `LinearMHDDriftkineticCC(turn_off=..., hot_epsilon=0.05)`. Measure the banana width and compare with $\propto\varepsilon$.
2. **Trapped–passing boundary:** add markers with $v_\parallel/v_\perp$ = 0.25, 0.30, 0.35. Where does the transition happen, and does it match $\sqrt{2r/R}$?
3. **Sampling:** in §4.2, try `loading="sobol_antithetic"` and compare the error returned by `show_distribution_function`.
4. **Passive vs active EPs:** in the smoke test, set `ep_scale=0.0` in all coupling propagators and compare `en_fv` and `en_fB` with the active run.
5. **Timing:** time `shearalfen_cc5d` alone at `dt=0.5` and `dt=0.1` (call `model.propagators.shearalfen_cc5d(dt)` after `sim.allocate()`) and relate it to your LinearMHD experience.
6. **Production:** submit `tae_ep/params_TAE_EP.py`, then a copy with `NEP_AXIS=0` (EP-free control), and compare γ and ω with the §9 toolkit.

---
## 12  Extension: a 15-minute demo of EP ↔ TAE coupling

**Question:** do the energetic ions exchange energy with the TAE, and does the wave grow because of them?

**Set-up** (`tae_ep_demo/params_demo.py`, submitted with `tae_ep_demo/run_demo.sh`): the ITPA-like torus and TAE seed from your `LinearMHD` run, plus EPs set up with everything from sections 3 to 5:
400 keV (vth ≈ 2.84), tanh density profile steepest at r = 0.5, `B0=equil.absB0`, δf with matched `moments`, explicit pushers, and the toroidal Fourier filter.
It is small and short: grid (16, 48, 4) with degree 2, 1e5 markers, dt = 0.5, t ≤ 60, 16 MPI ranks each.

**Three runs, identical except for the EPs' feedback on the wave:**

| run | `ep_scale` | EP density |
|---|---|---|
| passive | 0 (EPs feel the wave but do not push back) | n_EP |
| coupled | 1 | n_EP |
| coupled_3xnEP | 1 | 3 × n_EP |

The passive run is the reference: it has the same markers, the same seed and the same numerical errors. Any difference from it is caused by the EPs acting on the wave.
The model is linear, so the seed amplitude (0.3) is arbitrary. It is chosen large so the real response dominates the spurious δf from the non-equilibrium profile (§6.1).

In [ ]:
"""Plots for the short EP <-> TAE coupling demo (three runs made by run_demo.sh).

passive        : EPs feel the wave, but ep_scale=0 -> they do not act back (= the pure MHD wave)
coupled        : full two-way coupling, ITPA-like EP density
coupled_3xnEP  : full coupling with 3x the EP density
"""

import os

import h5py
import matplotlib.pyplot as plt
import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.path.abspath("tae_ep_demo")
RUNS = {"passive": "passive EPs (no feedback)", "coupled": "coupled, n_EP", "coupled_3xnEP": "coupled, 3 × n_EP"}
COLORS = {"passive": "0.35", "coupled": "#2a6fdb", "coupled_3xnEP": "#e8663d"}
FIT = {"passive": (0, 60), "coupled": (8, 20), "coupled_3xnEP": (4, 12)}  # early, clean exponential phase


def load(run):
    with h5py.File(os.path.join(HERE, run, "data", "data_proc0.hdf5"), "r") as f:
        S = {k: f["scalar"][k][:] for k in f["scalar"]}
        S["t"] = f["time/value"][:]
    S["E_wave"] = S["en_U"] + S["en_B"] + S["en_p"]
    S["E_EP"] = S["en_fv"] + S["en_fB"]
    return S


def growth_rate(t, E, t0, t1):
    """gamma from E ~ exp(2 gamma t) in [t0, t1]."""
    m = (t >= t0) & (t <= t1)
    slope, icpt = np.polyfit(t[m], np.log(E[m]), 1)
    return slope / 2, slope, icpt, m


S = {r: load(r) for r in RUNS}
fig, axs = plt.subplots(1, 3, figsize=(17, 4.4), layout="constrained")

# (a) wave energy on a log scale: straight line = exponential growth, slope = 2*gamma
for r, lab in RUNS.items():
    s = S[r]
    g, slope, icpt, m = growth_rate(s["t"], s["E_wave"], *FIT[r])
    axs[0].semilogy(s["t"], s["E_wave"], color=COLORS[r], label=f"{lab}:  γ = {g:.3f}")
    if r != "passive":
        axs[0].semilogy(s["t"][m], np.exp(icpt + slope * s["t"][m]), "k--", lw=1)
axs[0].set(xlabel="t  [code units]", ylabel="wave energy  en_U + en_B + en_p",
           title="(a) the TAE grows only when EPs act back")
axs[0].legend(fontsize=8)

# (b) energy transfer: change caused by the feedback = coupled run minus passive run (same markers, same seed)
p = S["passive"]
for r in ("coupled", "coupled_3xnEP"):
    s = S[r]
    n = min(len(s["t"]), len(p["t"]))
    t = s["t"][:n]
    axs[1].plot(t, s["E_wave"][:n] - p["E_wave"][:n], color=COLORS[r], label=f"wave gains  ({RUNS[r]})")
    axs[1].plot(t, s["E_EP"][:n] - p["E_EP"][:n], color=COLORS[r], ls="--", label=f"EPs gain  ({RUNS[r]})")
axs[1].axhline(0, color="0.6", lw=0.8)
axs[1].set(xlabel="t  [code units]", ylabel="energy change due to the feedback",
           title="(b) energy transfer: EPs lose, wave gains", ylim=(-0.8, 0.5))
axs[1].legend(fontsize=8)

# (c) the grown wave expels EPs -> removes its own drive (saturation)
for r, lab in RUNS.items():
    s = S[r]
    axs[2].plot(s["t"], s["n_lost_particles"] / 1e5 * 100, color=COLORS[r], label=lab)
axs[2].set(xlabel="t  [code units]", ylabel="markers lost at the wall  [% of Np]",
           title="(c) the large wave kicks EPs out")
axs[2].legend(fontsize=8)

os.makedirs(os.path.join(HERE, "figures"), exist_ok=True)
fig.savefig(os.path.join(HERE, "figures", "ep_tae_coupling_demo.png"), dpi=150)
plt.show()

for r in RUNS:
    s = S[r]
    g = growth_rate(s["t"], s["E_wave"], *FIT[r])[0]
    print(f"{r:14s} t_end={s['t'][-1]:5.1f}  E_wave: {s['E_wave'][0]:.2e} -> {s['E_wave'][-1]:.2e}  "
          f"gamma (fit t in {FIT[r]}) = {g:.3f}  lost = {100 * s['n_lost_particles'][-1] / 1e5:.1f}%")


**Reading the figure**

* **(a) Growth only with feedback.** With passive EPs the wave energy stays roughly flat (γ ≈ 0.003, a small numerical drift). With feedback it grows exponentially, with γ ≈ 0.05. With 3× the EPs it grows much faster (γ ≈ 0.19): **the drive comes from the EPs, and it grows with how many there are.**
* **(b) Energy transfer.** The coupled run minus the passive run shows the wave gaining energy while the EPs lose energy (dashed lines go negative). This is the direction expected for an EP-driven instability. The two curves are **not** equal and opposite: markers lost at the wall carry energy out, and the δf energy scalar has a sizeable numerical error (§6.1), so this panel is qualitative.
* **(c) Saturation by EP loss.** Once the wave is large it pushes EPs across the field and out to the wall (losses rise from about 5 % to over 30 %). The EPs that fed the wave are removed, so its growth slows. In (a) the 3 × n_EP curve bends over at t ≈ 15, exactly when its losses take off.

**What this is and what it isn't.** This is qualitative proof that the model's two-way EP–TAE coupling works: EPs lose energy, the wave grows, the growth scales with EP density, and the wave redistributes the EPs.
The numbers are **not** a TAE growth rate:
* γ/ω ≈ 0.5 is far above ITPA values of a few %.
* Grid, markers and run length are tiny.
* The large seed makes orbits respond nonlinearly.
* The tanh background is not an exact equilibrium.
* Late in the run en_U ≫ en_B, which is not the equal kinetic/magnetic split of a clean Alfvénic mode.

A quantitative γ needs the production file `tae_ep/params_TAE_EP.py`, a converged grid and marker count, a small seed, and control runs (see the plan in the notes).